# Complete Post-Training Pipeline
## Reward Modeling, PPO, KL Regularization, and DPO on UltraFeedback

**Model:** GPT-2 Small (124M parameters)
**Dataset:** UltraFeedback
**Hardware:** Google Colab T4 GPU (16GB VRAM)
**Framework:** HuggingFace TRL

## Project Notebook — Real Results

| Experiment | Result |
|---|---|
| GPU available | |
| GPU name | |
| Total VRAM | |
| GPT-2 load success | |
| DistilBERT load success | |
| UltraFeedback load success | |
| SFT loss | |
| Reward model accuracy | |
| PPO avg reward (with KL) | |
| PPO avg reward (no KL) | |
| DPO loss | |


In [ ]:
# ============================================================
# STAGE 0 — CELL 2: Install all libraries
# ============================================================
# We install everything the entire project needs right now.
# Reason: Installing mid-training wastes compute time and can
# cause kernel restarts that lose your GPU session.

# trl = Transformer Reinforcement Learning library by HuggingFace
# This handles SFT, PPO, DPO with a consistent API
# Without this we would have to implement PPO from scratch

# datasets = HuggingFace datasets library
# Lets us load UltraFeedback with one line of code

# accelerate = handles moving tensors between CPU and GPU cleanly
# peft = Parameter Efficient Fine Tuning — needed for LoRA in DPO

!pip install -q trl datasets accelerate peft

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 46.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STAGE 0 — CELL 3: Verify GPU availability
# ============================================================
import torch

# torch.cuda.is_available() returns True if a GPU is detected
# If this returns False, go to Runtime > Change Runtime Type > T4 GPU
print("GPU available:", torch.cuda.is_available())

# torch.cuda.get_device_name(0) returns the name of GPU number 0
# We should see "Tesla T4"
print("GPU name:", torch.cuda.get_device_name(0))

# torch.cuda.get_device_properties(0).total_memory gives total VRAM in bytes
# We divide by 1024^3 to convert bytes to gigabytes
total_memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"Total VRAM: {total_memory:.2f} GB")

# torch.cuda.memory_allocated(0) tells us how much VRAM is already used
# Right now nothing is loaded so this should be near 0
used_memory = torch.cuda.memory_allocated(0) / (1024**3)
print(f"Currently used VRAM: {used_memory:.3f} GB")
print(f"Available VRAM: {(total_memory - used_memory):.2f} GB")

GPU available: True
GPU name: Tesla T4
Total VRAM: 14.56 GB
Currently used VRAM: 0.000 GB
Available VRAM: 14.56 GB


 **Why we look at the first example:**

UltraFeedback has a specific structure. Before we write any code to extract chosen/rejected pairs, we need to see exactly what fields exist and what they're called. Never assume a dataset's structure — always look first.

In [ ]:
# ============================================================
# STAGE 0 — CELL 4: Load UltraFeedback dataset
# ============================================================
from datasets import load_dataset

# UltraFeedback is a real dataset used in research papers
# It contains prompts with multiple model responses and quality scores
# "train" split has ~60,000 examples. We only need a small subset.
# We load the full dataset object first, then we will slice it

print("Loading UltraFeedback dataset...")
dataset = load_dataset("openbmb/UltraFeedback", split="train")

print(f"Total examples in dataset: {len(dataset)}")

# Look at the first example to understand the structure
# This is critical — we need to know what fields are available
# before we write any processing code
print("\nFirst example keys:", dataset[0].keys())
print("\nFirst example:")
print(dataset[0])

Loading UltraFeedback dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/15.4k [00:00<?, ?B/s]

evol_instruct.jsonl:   0%|          | 0.00/168M [00:00<?, ?B/s]

false_qa.jsonl:   0%|          | 0.00/25.9M [00:00<?, ?B/s]

...

sharegpt.jsonl:   0%|          | 0.00/313M [00:00<?, ?B/s]

truthful_qa.jsonl:   0%|          | 0.00/9.99M [00:00<?, ?B/s]

ultrachat.jsonl:   0%|          | 0.00/182M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/63967 [00:00<?, ? examples/s]

Total examples in dataset: 63967

First example keys: dict_keys(['source', 'instruction', 'models', 'completions', 'correct_answers', 'incorrect_answers'])

First example:
{'source': 'evol_instruct', 'instruction': 'Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here\'s some starter code to help you out:\n#include <iostream>\n#include <string>\nusing namespace std;\nint main() {\n    string country;\n    // prompt user for input\n    cout << "Enter the name of a country: ";\n    cin >> country;\n    // check if country borders the Mediterranean Sea\n    // [C++ code]\n    return 0;\n}', 'models': ['alpaca-7b', 'pythia-12b', 'starchat', 'vicuna-33b'], 'completions': [{'annotations': {'helpfulness': {'Rating': '2', 'Rationale': 'The response is clear and not lengthy, but it lacks useful and comprehensive information.', 'Rationale For Rating': 'The code is partially incorrect as it checks if the country name

why cell 5 exists?
Why this cell exists:

UltraFeedback doesn't give you a simple "chosen" and "rejected" field. It gives you multiple responses with ratings. We need to understand this structure so we can write code to extract the best response (chosen) and worst response (rejected) from each example. This understanding directly affects Stages 1, 2, and 3.

In [ ]:
# ============================================================
# STAGE 0 — CELL 5: Understand UltraFeedback structure
# ============================================================

# UltraFeedback stores responses in a specific nested format
# Let's inspect one example carefully

example = dataset[0]

print("=== PROMPT ===")
print(example['instruction'])  # The question/instruction

print("\n=== NUMBER OF RESPONSES ===")
print(len(example['completions']))  # Multiple responses per prompt

print("\n=== FIRST RESPONSE ===")
print("Model:", example['completions'][0]['model'])
print("Response:", example['completions'][0]['response'][:200])  # First 200 chars

# Each completion has a rating — higher rating = better response
print("\n=== RATINGS OF ALL RESPONSES ===")
for i, completion in enumerate(example['completions']):
    rating = completion.get('annotations', {}).get('instruction_following', {}).get('Rating', 'N/A')
    print(f"Response {i}: Rating = {rating}")

=== PROMPT ===
Can you write a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea? Here's some starter code to help you out:
#include <iostream>
#include <string>
using namespace std;
int main() {
    string country;
    // prompt user for input
    cout << "Enter the name of a country: ";
    cin >> country;
    // check if country borders the Mediterranean Sea
    // [C++ code]
    return 0;
}

=== NUMBER OF RESPONSES ===
4

=== FIRST RESPONSE ===
Model: alpaca-7b
Response: int main() {
    string country;
    // prompt user for input
    cout << "Enter the name of a country: ";
    cin >> country;
    // check if country borders the Mediterranean Sea
    if (endsWith(co

=== RATINGS OF ALL RESPONSES ===
Response 0: Rating = 1
Response 1: Rating = 5
Response 2: Rating = 5
Response 3: Rating = 2


In [ ]:
# ============================================================
# STAGE 0 — CELL 6: Load GPT-2 Small and measure memory
# ============================================================
from transformers import GPT2LMHeadModel, GPT2Tokenizer

print("Loading GPT-2 Small tokenizer...")
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 tokenizer doesn't have a pad token by default
# We set it to eos_token (end of sequence token)
# Without this, batching multiple sequences causes errors
# because the model doesn't know how to handle sequences of different lengths
tokenizer.pad_token = tokenizer.eos_token

print("Loading GPT-2 Small model...")
model_gpt2 = GPT2LMHeadModel.from_pretrained("gpt2")

# Move model to GPU
# .to("cuda") means "put this model's parameters on the GPU"
# Without this, computation happens on CPU which is 10-50x slower
model_gpt2 = model_gpt2.to("cuda")

# Measure how much memory GPT-2 is using
memory_after_gpt2 = torch.cuda.memory_allocated(0) / (1024**3)
print(f"\nVRAM used after loading GPT-2: {memory_after_gpt2:.3f} GB")
print(f"GPT-2 model size: ~{memory_after_gpt2:.3f} GB")
print(f"Remaining VRAM: {(total_memory - memory_after_gpt2):.2f} GB")

Loading GPT-2 Small tokenizer...


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]


VRAM used after loading GPT-2: 0.464 GB
GPT-2 model size: ~0.464 GB
Remaining VRAM: 14.10 GB


In [ ]:
# ============================================================
# STAGE 0 — CELL 7: Load DistilBERT and check combined memory
# ============================================================
from transformers import DistilBertModel, DistilBertTokenizer

print("Loading DistilBERT tokenizer...")
tokenizer_distilbert = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

print("Loading DistilBERT model...")
model_distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
model_distilbert = model_distilbert.to("cuda")

# Measure combined memory
memory_after_both = torch.cuda.memory_allocated(0) / (1024**3)
print(f"\nVRAM used after loading both models: {memory_after_both:.3f} GB")
print(f"DistilBERT alone: ~{(memory_after_both - memory_after_gpt2):.3f} GB")
print(f"Remaining VRAM for training: {(total_memory - memory_after_both):.2f} GB")

# Safety check — we need at least 8GB free for PPO (which needs 4 models)
if (total_memory - memory_after_both) > 8:
    print("\n MEMORY CHECK PASSED — Sufficient VRAM for this project")
else:
    print("\nWARNING — Low VRAM. May need to reduce batch sizes")

Loading DistilBERT tokenizer...


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Loading DistilBERT model...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



VRAM used after loading both models: 0.713 GB
DistilBERT alone: ~0.248 GB
Remaining VRAM for training: 13.85 GB

✅ MEMORY CHECK PASSED — Sufficient VRAM for this project


In [ ]:
# ============================================================
# STAGE 0 — CELL 8: Forward pass test on 100 examples
# ============================================================

# Take 100 examples from UltraFeedback
test_examples = dataset.select(range(100))

# Extract just the instruction text from each example
# We'll process the full data structure properly in Stage 1
# Right now we just want to check that tokenization and forward pass work
test_texts = [ex['instruction'] for ex in test_examples]

print(f"Testing forward pass on {len(test_texts)} examples...")
print(f"Sample text: {test_texts[0][:100]}...")

# Tokenize — convert text into numbers the model understands
# padding=True: make all sequences same length with pad tokens
# truncation=True: cut sequences longer than max_length
# max_length=128: we use short length for feasibility test only
# return_tensors="pt": return PyTorch tensors (not numpy arrays)
inputs = tokenizer(
    test_texts[:10],       # Just 10 at a time to be safe
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Move tokenized inputs to GPU
inputs = {key: val.to("cuda") for key, val in inputs.items()}

# Run forward pass
# torch.no_grad() means "don't compute gradients right now"
# We use this during inference/testing to save memory
# Gradients are only needed during training
with torch.no_grad():
    outputs = model_gpt2(**inputs)

print(f"\n Forward pass successful!")
print(f"Input shape: {inputs['input_ids'].shape}")
print(f"Output logits shape: {outputs.logits.shape}")

# Check memory after forward pass
memory_after_forward = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM after forward pass: {memory_after_forward:.3f} GB")
print(f"Remaining VRAM: {(total_memory - memory_after_forward):.2f} GB")

Testing forward pass on 100 examples...
Sample text: Can you write a C++ program that prompts the user to enter the name of a country and checks if it bo...

✅ Forward pass successful!
Input shape: torch.Size([10, 128])
Output logits shape: torch.Size([10, 128, 50257])
VRAM after forward pass: 1.055 GB
Remaining VRAM: 13.51 GB


In [ ]:
# ============================================================
# STAGE 0 — CELL 9: Feasibility report
# ============================================================

print("=" * 50)
print("STAGE 0 FEASIBILITY REPORT")
print("=" * 50)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Total VRAM: {total_memory:.2f} GB")
print(f"GPT-2 Small size: {memory_after_gpt2:.3f} GB")
print(f"DistilBERT size: {(memory_after_both - memory_after_gpt2):.3f} GB")
print(f"Both models loaded: {memory_after_both:.3f} GB")
print(f"After forward pass: {memory_after_forward:.3f} GB")
print(f"Free VRAM remaining: {(total_memory - memory_after_forward):.2f} GB")
print(f"UltraFeedback total examples: {len(dataset)}")
print(f"Forward pass test: PASSED")
print("=" * 50)

# Clean up — remove models from GPU memory
# We will reload them properly in each stage
# del removes the Python object
# torch.cuda.empty_cache() releases the GPU memory back

# Check if model_gpt2 exists in the global scope before trying to delete it
if 'model_gpt2' in globals():
    del model_gpt2
# Check if model_distilbert exists in the global scope before trying to delete it
if 'model_distilbert' in globals():
    del model_distilbert
torch.cuda.empty_cache()

memory_after_cleanup = torch.cuda.memory_allocated(0) / (1024**3)
print(f"\nVRAM after cleanup: {memory_after_cleanup:.3f} GB")
print(" Stage 0 Complete. Ready to begin Stage 1.")

STAGE 0 FEASIBILITY REPORT
GPU: Tesla T4
Total VRAM: 14.56 GB
GPT-2 Small size: 0.464 GB
DistilBERT size: 0.248 GB
Both models loaded: 0.713 GB
After forward pass: 1.055 GB
Free VRAM remaining: 13.51 GB
UltraFeedback total examples: 63967
Forward pass test: PASSED

VRAM after cleanup: 0.342 GB
 Stage 0 Complete. Ready to begin Stage 1.


*UltraFeedback* is a preference dataset where each example contains an instruction and multiple model-generated responses to that instruction. These responses were generated by different LLMs and then evaluated for qualities such as helpfulness, truthfulness, instruction-following, and overall quality.

Structurally, each example looks like:

Instruction
├── Completion 1 + scores
├── Completion 2 + scores
├── Completion 3 + scores
└── Completion 4 + scores

Each completion has an overall_score, which summarizes its quality. To construct preference pairs, I compared the scores of all completions for the same instruction.

For example, if the scores were:

Response A → 1
Response B → 5
Response C → 4
Response D → 2

then I selected one of the highest-scoring responses as the chosen response and one of the lowest-scoring responses as the rejected response.

So the resulting preference pair becomes:

Prompt: Explain gravity to a 10-year-old.

Chosen:
Gravity is an invisible pull that keeps us on Earth...

Rejected:
Gravity is a manifestation of spacetime curvature...

The intuition is that the chosen response reflects what humans preferred, while the rejected response reflects a lower-quality alternative for the same prompt.

I then used these extracted pairs in two ways:

For reward model training, where the model learns to assign a higher reward to the chosen response than to the rejected response.
For DPO training, where the policy is directly optimized to prefer the chosen response over the rejected one.

For SFT, I only used the chosen responses because SFT requires demonstrations of good behavior rather than comparisons between good and bad behavior.

One reason I selected UltraFeedback is that the same dataset can support the entire post-training pipeline—SFT, reward modeling, PPO, and DPO—without introducing dataset mismatches between stages.

"Stage 0 was a feasibility and data-understanding stage. I verified that GPT-2 and DistilBERT fit comfortably on a T4 GPU, inspected the UltraFeedback dataset structure, understood how chosen and rejected responses are derived from completion scores, and confirmed that the same dataset could support SFT, reward-model training, PPO, and DPO without any dataset mismatch issues."

# Stage 1 — Dataset Loading and Understanding

##  Extract Chosen and Rejected from UltraFeedback

For every example in UltraFeedback:

Look at all completions
Find the one with the highest overall_score → that's chosen

Find the one with the lowest overall_score → that's rejected

If there's a tie in overall_score → use

fine_grained_score to break it

Make sure chosen and rejected are not the same response

Return the instruction, chosen response, rejected
response, and their scores

In [ ]:
# ============================================================
# STAGE 1 — CELL 1: Extract chosen and rejected pairs
# ============================================================

# We need this to work with the dataset we loaded in Stage 0
# dataset is still in memory from Stage 0 Cell 4
# If you restarted the kernel, re-run Stage 0 Cell 4 first

def extract_preference_pair(example):
    """
    Takes one UltraFeedback example.
    Returns chosen response, rejected response, and their scores.
    Returns None if we cannot extract a valid pair.
    """

    completions = example['completions']

    # Safety check — we need at least 2 completions to make a pair
    # If an example has only 1 completion, skip it
    if len(completions) < 2:
        return None

    # Score each completion
    # We collect (overall_score, fine_grained_score, response_text) for each
    scored = []
    for completion in completions:

        # overall_score is the primary ranking number
        # It might be missing in some examples so we use .get() with default 0
        overall = completion.get('overall_score', 0)

        # fine_grained_score is the average of 4 annotation dimensions
        # Used only as tiebreaker when overall_score is equal
        fine_grained = completion.get('fine-grained_score', 0)

        # The actual text response
        response = completion.get('response', '')

        # Skip empty responses — a blank response is useless for training
        if not response.strip():
            continue

        scored.append((overall, fine_grained, response))

    # After filtering empty responses, check we still have at least 2
    if len(scored) < 2:
        return None

    # Sort by overall_score first (descending = highest first)
    # If overall_score is equal, sort by fine_grained_score (descending)
    # Python's sort is stable and handles tuples left to right
    # So (overall, fine_grained) sorting does exactly what we want
    scored.sort(key=lambda x: (x[0], x[1]), reverse=True)

    # Best response = first after sorting = chosen
    chosen_overall, chosen_fine, chosen_response = scored[0]

    # Worst response = last after sorting = rejected
    rejected_overall, rejected_fine, rejected_response = scored[-1]

    # Make sure chosen and rejected are actually different responses
    # This can happen if all completions are identical text
    if chosen_response == rejected_response:
        return None

    # Make sure there is actually a score difference
    # If all responses have the same score, the pair is not meaningful
    if chosen_overall == rejected_overall and chosen_fine == rejected_fine:
        return None

    return {
        'instruction': example['instruction'],
        'chosen': chosen_response,
        'rejected': rejected_response,
        'chosen_score': chosen_overall,
        'rejected_score': rejected_overall,
        'score_gap': chosen_overall - rejected_overall
    }

# Test the function on the first example before applying to full dataset
test_result = extract_preference_pair(dataset[0])

print("=== TEST ON FIRST EXAMPLE ===")
print(f"\nInstruction: {test_result['instruction'][:100]}...")
print(f"\nChosen (first 200 chars): {test_result['chosen'][:200]}")
print(f"\nRejected (first 200 chars): {test_result['rejected'][:200]}")
print(f"\nChosen score:   {test_result['chosen_score']}")
print(f"Rejected score: {test_result['rejected_score']}")
print(f"Score gap:      {test_result['score_gap']}")

# ============================================================
# PSEUDOCODE SUMMARY:
# for each example:
#     collect (overall_score, fine_grained_score, response) for all completions
#     skip completions with empty responses
#     sort completions by (overall_score, fine_grained_score) descending
#     chosen   = highest scoring completion
#     rejected = lowest scoring completion
#     if chosen == rejected or no score gap → skip example
#     return instruction, chosen, rejected, scores, score_gap
# ============================================================

=== TEST ON FIRST EXAMPLE ===

Instruction: Can you write a C++ program that prompts the user to enter the name of a country and checks if it bo...

Chosen (first 200 chars): Here's a C++ program that prompts the user to enter the name of a country and checks if it borders the Mediterranean Sea:

#include <iostream>
#include <string>
#include <set>
#include <map>
#include 

Rejected (first 200 chars): Sure, here is the program using the C++11 algorithm "cds::algorithm::GreaterEqual":
#include <iostream>
#include <string>
#include <algorithm>
#include <vector>
#include <cctype>

using namespace std;

Chosen score:   7.5
Rejected score: 3.0
Score gap:      4.5


In [ ]:
# ============================================================
# STAGE 1 — CELL 2: Apply extraction to full dataset
# ============================================================

print("Extracting preference pairs from full dataset...")
print("This may take 1-2 minutes...")

# Apply our function to every example in the dataset
# We collect results and filter out None values
# (None means the example was skipped — empty response or no score gap)

all_pairs = []

for i, example in enumerate(dataset):

    result = extract_preference_pair(example)

    # Only keep valid pairs
    if result is not None:
        all_pairs.append(result)

    # Print progress every 10000 examples so we know it's running
    if (i + 1) % 10000 == 0:
        print(f"Processed {i+1}/{len(dataset)} examples. Valid pairs so far: {len(all_pairs)}")

print(f"\nTotal examples in dataset:    {len(dataset)}")
print(f"Valid preference pairs found: {len(all_pairs)}")
print(f"Examples skipped:             {len(dataset) - len(all_pairs)}")

# Look at score gap distribution
# This tells us how meaningful our preference pairs are
import numpy as np

score_gaps = [pair['score_gap'] for pair in all_pairs]
print(f"\nScore gap statistics:")
print(f"  Average gap:  {np.mean(score_gaps):.2f}")
print(f"  Minimum gap:  {np.min(score_gaps):.2f}")
print(f"  Maximum gap:  {np.max(score_gaps):.2f}")
print(f"  Pairs with gap > 2.0: {sum(1 for g in score_gaps if g > 2.0)}")
print(f"  Pairs with gap > 4.0: {sum(1 for g in score_gaps if g > 4.0)}")

# ============================================================
# PSEUDOCODE SUMMARY:
# for each example in full dataset:
#     call extract_preference_pair(example)
#     if result is not None → add to all_pairs list
# compute score gap statistics across all valid pairs
# report how many examples were kept vs skipped
# ============================================================

Extracting preference pairs from full dataset...
This may take 1-2 minutes...
Processed 10000/63967 examples. Valid pairs so far: 9990
Processed 20000/63967 examples. Valid pairs so far: 19970
Processed 30000/63967 examples. Valid pairs so far: 29905
Processed 40000/63967 examples. Valid pairs so far: 39875
Processed 50000/63967 examples. Valid pairs so far: 49857
Processed 60000/63967 examples. Valid pairs so far: 59842

Total examples in dataset:    63967
Valid preference pairs found: 63806
Examples skipped:             161

Score gap statistics:
  Average gap:  3.02
  Minimum gap:  0.00
  Maximum gap:  9.00
  Pairs with gap > 2.0: 35002
  Pairs with gap > 4.0: 17595


Only 161 examples were skipped out of 63,967. That means UltraFeedback is extremely clean data. Almost every example had at least 2 valid responses with a score difference. 161 skipped means either empty responses or all responses had identical scores.

 on average the chosen response scores 3 points higher than the rejected response. That is a meaningful difference. Our reward model will have clear signal to learn from.

Some pairs have zero gap. Chosen and rejected have the exact same overall_score. These are useless for training.

 If both responses are equally good or equally bad, what does the reward model learn from that? Nothing. We will filter these out in the next cell.

35,002 pairs where the quality difference is meaningful

17,595 pairs where the quality difference is very obvious and strong

In [ ]:
# ============================================================
# STAGE 1 — CELL 3: Filter and create final splits
# ============================================================
import random
random.seed(42)
# random.seed(42) means: fix the randomness so results are reproducible
# If you run this again tomorrow you get the exact same split
# Without this, every run gives different splits which makes
# experiments impossible to compare

# Step 1 — Filter out pairs with zero score gap
# Zero gap means chosen and rejected are equally scored
# These give no learning signal to the reward model
meaningful_pairs = [p for p in all_pairs if p['score_gap'] > 0]
print(f"Pairs with score gap > 0: {len(meaningful_pairs)}")

# Step 2 — Further filter to gap > 2.0 for stronger signal
strong_pairs = [p for p in all_pairs if p['score_gap'] > 2.0]
print(f"Pairs with score gap > 2.0: {len(strong_pairs)}")

# Step 3 — Shuffle so we don't accidentally pick examples
# from only one source (evol_instruct, flan, sharegpt etc)
random.shuffle(strong_pairs)

# Step 4 — Create our three splits
# 2000 chosen responses for SFT
# 1000 preference pairs for reward model and DPO
# 200 held out for evaluation

sft_data = strong_pairs[:2000]
preference_data = strong_pairs[2000:3000]
eval_data = strong_pairs[3000:3200]

print(f"\nSplit sizes:")
print(f"  SFT data (chosen responses only): {len(sft_data)}")
print(f"  Preference pairs (reward model + DPO): {len(preference_data)}")
print(f"  Evaluation set: {len(eval_data)}")
print(f"  Total used: {len(sft_data) + len(preference_data) + len(eval_data)}")
print(f"  Total available: {len(strong_pairs)}")
print(f"  Unused (buffer): {len(strong_pairs) - len(sft_data) - len(preference_data) - len(eval_data)}")

# Step 5 — Verify no overlap between splits
# This is critical — if the same example appears in both
# SFT training and evaluation, our eval results are meaningless
sft_instructions = set(d['instruction'] for d in sft_data)
pref_instructions = set(d['instruction'] for d in preference_data)
eval_instructions = set(d['instruction'] for d in eval_data)

sft_pref_overlap = sft_instructions.intersection(pref_instructions)
sft_eval_overlap = sft_instructions.intersection(eval_instructions)
pref_eval_overlap = pref_instructions.intersection(eval_instructions)

print(f"\nOverlap check:")
print(f"  SFT ∩ Preference: {len(sft_pref_overlap)} examples")
print(f"  SFT ∩ Eval: {len(sft_eval_overlap)} examples")
print(f"  Preference ∩ Eval: {len(pref_eval_overlap)} examples")

if len(sft_pref_overlap) == 0 and len(sft_eval_overlap) == 0 and len(pref_eval_overlap) == 0:
    print("\n No overlap between splits. Clean dataset.")
else:
    print("\n WARNING: Overlap detected. Investigate before proceeding.")

# ============================================================
# PSEUDOCODE SUMMARY:
# filter all_pairs where score_gap > 0 → meaningful_pairs
# filter all_pairs where score_gap > 2.0 → strong_pairs
# shuffle strong_pairs with fixed random seed 42
# sft_data       = strong_pairs[0:2000]
# preference_data = strong_pairs[2000:3000]
# eval_data      = strong_pairs[3000:3200]
# verify zero overlap between all three splits
# ============================================================

Pairs with score gap > 0: 63121
Pairs with score gap > 2.0: 35002

Split sizes:
  SFT data (chosen responses only): 2000
  Preference pairs (reward model + DPO): 1000
  Evaluation set: 200
  Total used: 3200
  Total available: 35002
  Unused (buffer): 31802

Overlap check:
  SFT ∩ Preference: 0 examples
  SFT ∩ Eval: 0 examples
  Preference ∩ Eval: 0 examples

 No overlap between splits. Clean dataset.


Unused buffer: 31,802 — we are using only 3,200 out of 35,002 strong pairs. This means if anything goes wrong we have a massive backup pool to draw from without touching eval data.

Overlap: 0 everywhere — clean. No data leakage anywhere. SFT training data, preference pairs, and evaluation set are completely separate. This means our evaluation results will be honest.

In [ ]:
# ============================================================
# STAGE 1 — CELL 4: Understand what we have
# ============================================================
import numpy as np

print("=" * 55)
print("DATASET UNDERSTANDING REPORT")
print("=" * 55)

# --- Chosen response lengths ---
# Length in characters tells us how long our SFT training
# examples are. Very short responses may be low quality.
# Very long responses may exceed GPT-2's context window (1024 tokens)
chosen_lengths = [len(p['chosen']) for p in sft_data]
rejected_lengths = [len(p['rejected']) for p in sft_data]

print(f"\nChosen response lengths (characters):")
print(f"  Average: {np.mean(chosen_lengths):.0f}")
print(f"  Minimum: {np.min(chosen_lengths):.0f}")
print(f"  Maximum: {np.max(chosen_lengths):.0f}")

print(f"\nRejected response lengths (characters):")
print(f"  Average: {np.mean(rejected_lengths):.0f}")
print(f"  Minimum: {np.min(rejected_lengths):.0f}")
print(f"  Maximum: {np.max(rejected_lengths):.0f}")

# --- Score distributions ---
chosen_scores = [p['chosen_score'] for p in sft_data]
rejected_scores = [p['rejected_score'] for p in sft_data]
gaps = [p['score_gap'] for p in sft_data]

print(f"\nChosen scores (our SFT training data):")
print(f"  Average: {np.mean(chosen_scores):.2f}")
print(f"  Minimum: {np.min(chosen_scores):.2f}")
print(f"  Maximum: {np.max(chosen_scores):.2f}")

print(f"\nRejected scores:")
print(f"  Average: {np.mean(rejected_scores):.2f}")
print(f"  Minimum: {np.min(rejected_scores):.2f}")
print(f"  Maximum: {np.max(rejected_scores):.2f}")

print(f"\nScore gaps in our SFT split:")
print(f"  Average gap: {np.mean(gaps):.2f}")

# --- Show 3 side by side comparisons ---
print("\n" + "=" * 55)
print("3 EXAMPLE PAIRS SIDE BY SIDE")
print("=" * 55)

for i in range(3):
    pair = preference_data[i]
    print(f"\n--- Example {i+1} ---")
    print(f"Instruction: {pair['instruction'][:120]}...")
    print(f"\nCHOSEN (score {pair['chosen_score']}):")
    print(pair['chosen'][:300])
    print(f"\nREJECTED (score {pair['rejected_score']}):")
    print(pair['rejected'][:300])
    print(f"\nScore gap: {pair['score_gap']}")
    print("-" * 55)

# ============================================================
# PSEUDOCODE SUMMARY:
# compute length statistics for chosen and rejected responses
# compute score statistics for chosen and rejected responses
# print 3 side by side chosen vs rejected comparisons
# goal: understand what good vs bad looks like in our data
# ============================================================

DATASET UNDERSTANDING REPORT

Chosen response lengths (characters):
  Average: 1044
  Minimum: 1
  Maximum: 5431

Rejected response lengths (characters):
  Average: 692
  Minimum: 1
  Maximum: 4985

Chosen scores (our SFT training data):
  Average: 7.91
  Minimum: 4.00
  Maximum: 10.00

Rejected scores:
  Average: 3.48
  Minimum: 1.00
  Maximum: 7.00

Score gaps in our SFT split:
  Average gap: 4.43

3 EXAMPLE PAIRS SIDE BY SIDE

--- Example 1 ---
Instruction: Can you use your reasoning skills to complete this puzzle? Fill in the blanks with three letters to create a word relate...

CHOSEN (score 8.0):
Yes, I can help you complete the puzzle. The word you are looking for is "AID." The news article title would be "AI Aid Help Create a Vaccine for Coronavirus." "Aid" is a synonym for "assist."

REJECTED (score 4.0):
Hello! I'm here to help you with your question. I understand that you want me to complete a puzzle related to a news article about AI assisting in creating a vaccine for Coro

In [ ]:
# ============================================================
# STAGE 1 — CELL 5: Save splits to disk
# ============================================================
import json

# Save SFT data — we only need instruction and chosen response
# We don't need rejected for SFT because SFT only trains on good examples
sft_save = [{'instruction': p['instruction'],
              'chosen': p['chosen'],
              'score': p['chosen_score']} for p in sft_data]

with open('sft_data.json', 'w') as f:
    json.dump(sft_save, f)

print(f"Saved SFT data: {len(sft_save)} examples → sft_data.json")

# Save preference pairs — we need instruction, chosen, AND rejected
# This is used for reward model training and DPO
pref_save = [{'instruction': p['instruction'],
               'chosen': p['chosen'],
               'rejected': p['rejected'],
               'chosen_score': p['chosen_score'],
               'rejected_score': p['rejected_score'],
               'score_gap': p['score_gap']} for p in preference_data]

with open('preference_data.json', 'w') as f:
    json.dump(pref_save, f)

print(f"Saved preference data: {len(pref_save)} pairs → preference_data.json")

# Save eval data — same structure as preference data
# We save both chosen and rejected so we can test the reward model
# on both directions
eval_save = [{'instruction': p['instruction'],
               'chosen': p['chosen'],
               'rejected': p['rejected'],
               'chosen_score': p['chosen_score'],
               'rejected_score': p['rejected_score'],
               'score_gap': p['score_gap']} for p in eval_data]

with open('eval_data.json', 'w') as f:
    json.dump(eval_save, f)

print(f" Saved eval data: {len(eval_save)} examples → eval_data.json")

# Verify files were saved correctly by loading them back
sft_check = json.load(open('sft_data.json'))
pref_check = json.load(open('preference_data.json'))
eval_check = json.load(open('eval_data.json'))

print(f"\nVerification — files loaded back successfully:")
print(f"  sft_data.json:        {len(sft_check)} examples")
print(f"  preference_data.json: {len(pref_check)} pairs")
print(f"  eval_data.json:       {len(eval_check)} examples")

# ============================================================
# PSEUDOCODE SUMMARY:
# sft_data → keep only instruction + chosen + score → save as sft_data.json
# preference_data → keep instruction + chosen + rejected + scores → save as preference_data.json
# eval_data → same structure as preference → save as eval_data.json
# reload all three files to verify they saved correctly
# ============================================================

Saved SFT data: 2000 examples → sft_data.json
Saved preference data: 1000 pairs → preference_data.json
 Saved eval data: 200 examples → eval_data.json

Verification — files loaded back successfully:
  sft_data.json:        2000 examples
  preference_data.json: 1000 pairs
  eval_data.json:       200 examples


I chose a threshold of gap > 2 because it balances label quality and dataset size. If the threshold is too low, such as 0.5, the chosen and rejected responses become very similar and the preference labels become noisy. The reward model learns from ambiguous comparisons and may generalize poorly.

If the threshold is too high, such as 7, the reward model only sees extremely easy comparisons. Training becomes cleaner, but the model may fail to learn the subtle distinctions that matter in realistic post-training scenarios.

A threshold around 2 preserves a strong preference signal while still exposing the model to a range of comparison difficulties.

# Stage 2 — SFT on Chosen Responses

We are taking GPT-2 Small and fine-tuning it on our 2000 chosen responses from UltraFeedback.

 After this stage, GPT-2 will stop behaving like a random text predictor and start behaving like a model that follows instructions and gives decent responses.

 This is called Supervised Fine-Tuning — SFT.

Before SFT, if you give GPT-2 the prompt "Explain gravity to a 10 year old" — it might continue with anything. Another question. Random text.

 A news article. It doesn't know it's supposed to answer.

After SFT, GPT-2 has seen 2000 examples of the format:

Instruction: [question]

Response: [good answer]


It learns that when it sees an instruction, it should produce a response in the style of the good answers it was trained on.

The SFT model we produce here is not just the starting point for Stage 2. It is the reference policy for everything downstream.

- PPO measures how far the current model drifts from this SFT model using KL penalty
- DPO anchors its loss calculation to this SFT model's probability distributions
- GRPO also starts from this model

If SFT is bad — everything after is bad. This is the foundation.

In [ ]:
# ============================================================
# STAGE 2 — CELL 1: Format SFT training data
# ============================================================
import json

# Load the SFT data we saved in Stage 1
with open('sft_data.json', 'r') as f:
    sft_data = json.load(f)

print(f"Loaded {len(sft_data)} SFT examples")

# We need to format each example into a single string
# that GPT-2 will learn to continue
# The format is:
# ### Instruction: [the question]
# ### Response: [the good answer] [EOS token]
#
# Why this exact format?
# 1. ### markers make it easy for the model to learn
#    where instruction ends and response begins
# 2. EOS token at the end tells the model "stop here"
#    without EOS the model learns to generate forever
# 3. Consistent format means the model learns ONE pattern
#    not many different patterns

def format_sft_example(example):
    """
    Takes one SFT example with instruction and chosen response.
    Returns a formatted string ready for GPT-2 training.
    """
    instruction = example['instruction'].strip()
    response = example['chosen'].strip()

    # This is the template every example will follow
    formatted = f"### Instruction:\n{instruction}\n\n### Response:\n{response}"

    return formatted

# Apply formatting to all 2000 examples
formatted_texts = [format_sft_example(ex) for ex in sft_data]

# Check what the formatted data looks like
print("\n=== FORMATTED EXAMPLE ===")
print(formatted_texts[0][:500])
print("...")

# Check length distribution after formatting
lengths = [len(t) for t in formatted_texts]
import numpy as np
print(f"\nFormatted text lengths:")
print(f"  Average: {np.mean(lengths):.0f} characters")
print(f"  Maximum: {np.max(lengths):.0f} characters")
print(f"  Examples over 4000 chars: {sum(1 for l in lengths if l > 4000)}")

# ============================================================
# PSEUDOCODE SUMMARY:
# load sft_data.json
# for each example:
#     format as "### Instruction:\n{instruction}\n\n### Response:\n{response}"
# collect all formatted strings into formatted_texts list
# check length distribution to verify fit within GPT-2 context window
# ============================================================

Loaded 2000 SFT examples

=== FORMATTED EXAMPLE ===
### Instruction:
From: Reed Hastings
Sent: Sunday, August 14, 2016 6:36 PM
To: Peter Thiel
Subject: straightforward

Peter,

I appreciate that we can disagree and be direct with each other. I have our board Gordy feedback session tomorrow. I see our board being about great judgment, particularly in unlikely disaster where we have to pick new leaders. I'm so mystified by your endorsement of Trump for our President, that for me it moves from "different judgment" to "bad judgment". Some diversity i
...

Formatted text lengths:
  Average: 1765 characters
  Maximum: 8236 characters
  Examples over 4000 chars: 138


138 out of 2000 examples exceeded GPT-2's 1024 token context window and got truncated during tokenization.

> Add blockquote




This means the model trained on incomplete responses for 7% of the data, which can affect the model's ability to generate clean endings.

In production I would filter these examples out before training or use a model with a larger context window. For this experiment the impact was acceptable given the 93% of clean complete examples.

Average: 1765 characters — this is fine. GPT-2 handles up to ~4096 characters. Our average is well within limit.

Examples over 4000 chars: 138 — 138 out of 2000 examples will get partially cut off. That's 7% of our training data. Not catastrophic but worth knowing.


In [ ]:
# ============================================================
# STAGE 2 — CELL 2: Create dataset object and load tokenizer
# ============================================================
from datasets import Dataset
from transformers import GPT2Tokenizer

# Convert our list of formatted strings into a HuggingFace Dataset object
# TRL's SFTTrainer expects a HuggingFace Dataset, not a plain Python list
# We create a dictionary with key "text" containing all our formatted strings
hf_dataset = Dataset.from_dict({"text": formatted_texts})

print(f"HuggingFace Dataset created: {len(hf_dataset)} examples")
print(f"Dataset features: {hf_dataset.features}")

# Split into train and validation
# 1800 examples for training, 200 for validation
# Validation set lets us track if the model is overfitting
# Overfitting = model memorizes training data but fails on new examples
hf_dataset = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = hf_dataset['train']
val_dataset = hf_dataset['test']

print(f"\nTrain examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

# Load GPT-2 tokenizer
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# GPT-2 has no pad token by default
# We set it to eos_token — same fix as Stage 0
# Without this batching fails because sequences have different lengths
tokenizer.pad_token = tokenizer.eos_token

print(f"\nTokenizer vocabulary size: {tokenizer.vocab_size}")
print(f"EOS token: '{tokenizer.eos_token}' (id: {tokenizer.eos_token_id})")
print(f"PAD token: '{tokenizer.pad_token}' (id: {tokenizer.pad_token_id})")

# Quick check — tokenize one example and see how many tokens it produces
sample_tokens = tokenizer(formatted_texts[0], truncation=True, max_length=1024)
print(f"\nFirst example token count: {len(sample_tokens['input_ids'])}")
print(f"(Truncated to 1024 maximum)")

# ============================================================
# PSEUDOCODE SUMMARY:
# convert formatted_texts list → HuggingFace Dataset object
# split 90% train / 10% validation with seed 42
# load GPT2 tokenizer
# set pad_token = eos_token
# verify by tokenizing one example
# ============================================================

HuggingFace Dataset created: 2000 examples
Dataset features: {'text': Value('string')}

Train examples: 1800
Validation examples: 200

Tokenizer vocabulary size: 50257
EOS token: '<|endoftext|>' (id: 50256)
PAD token: '<|endoftext|>' (id: 50256)

First example token count: 299
(Truncated to 1024 maximum)


EOS token and PAD token are the same: <|endoftext|> — this confirms our fix worked. Both end-of-sequence and padding use the same token id 50256.


Train: 1800, Validation: 200 — 90/10 split worked correctly.


In [ ]:
# ============================================================
# STAGE 2 — CELL 3: Load GPT-2 and run SFT training
# ============================================================
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from trl import SFTConfig, SFTTrainer

# Load GPT-2 Small
# GPT2LMHeadModel = GPT-2 with a language modeling head on top
# "LM head" means: a linear layer that converts hidden states
# into probability scores over the vocabulary (50257 tokens)
model = GPT2LMHeadModel.from_pretrained("gpt2")
model = model.to("cuda")

print(f"GPT-2 loaded")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Count trainable parameters
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,}")

# Check memory after loading
memory_used = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM used after loading GPT-2: {memory_used:.3f} GB")

# ============================================================
# SFT CONFIGURATION
# Every number here is a deliberate choice — explained below
# ============================================================
sft_config = SFTConfig(

    output_dir="./sft_model",
    # Where to save checkpoints and final model
    # If Colab crashes during training, we can resume from here

    num_train_epochs=3,
    # How many times to go through all 1800 training examples
    # 1 epoch = model sees all 1800 examples once
    # 3 epochs = model sees each example 3 times
    # Too few epochs = model doesn't learn enough (underfitting)
    # Too many epochs = model memorizes training data (overfitting)
    # 3 is the standard starting point for SFT

    per_device_train_batch_size=4,
    # How many examples to process simultaneously in one step
    # Larger batch = more stable gradients but more memory
    # Smaller batch = noisier gradients but less memory
    # 4 is safe for T4 with GPT-2 Small

    per_device_eval_batch_size=4,
    # Same as above but for validation — no gradient computation
    # so this could be larger, but we keep it same for simplicity

    gradient_accumulation_steps=4,
    # This is a memory saving trick
    # Instead of batch size 16 (which might OOM),
    # we process 4 examples × 4 steps = effectively batch size 16
    # Gradients accumulate across 4 steps before one weight update
    # Same mathematical effect as batch size 16, half the memory

    learning_rate=2e-5,
    # How big each weight update step is
    # 2e-5 means 0.00002
    # Standard learning rate for fine-tuning language models
    # Too high = training becomes unstable, loss explodes
    # Too low = training is too slow, model barely changes

    warmup_steps=100,
    # For first 100 steps, learning rate gradually increases
    # from 0 to 2e-5 instead of jumping straight to 2e-5
    # Why? Sudden large learning rate at start can destabilize
    # the model before it has adjusted to the new data

    logging_steps=50,
    # Print training loss every 50 steps
    # Lets us watch if training is progressing

    eval_strategy="epoch",
    # Run validation at the end of every epoch
    # This gives us 3 validation measurements (one per epoch)
    # to track if model is improving or overfitting

    save_strategy="epoch",
    # Save model checkpoint at end of every epoch
    # If training crashes at epoch 2, we can resume from epoch 1

    load_best_model_at_end=True,
    # After all epochs finish, load whichever checkpoint
    # had the best validation loss
    # Protects against overfitting in later epochs

    max_length=1024,
    # Maximum token length — matches GPT-2's context window
    # Examples longer than this get truncated

    dataset_text_field="text",
    # Tell SFTTrainer which column in our dataset contains the text
    # Our dataset has one column called "text" — this matches

    report_to="none",
    # Don't send logs to wandb or other tracking services
    # Keeps things simple for our experiment
)

# Create the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("\nSFTTrainer created successfully")
print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")
print(f"Steps per epoch: {len(train_dataset) // sft_config.per_device_train_batch_size}")
print(f"Total training steps: {len(train_dataset) // sft_config.per_device_train_batch_size * sft_config.num_train_epochs}")

# ============================================================
# PSEUDOCODE SUMMARY:
# load GPT-2 Small → move to GPU
# configure SFTTrainer:
#     3 epochs, batch size 4, gradient accumulation 4
#     learning rate 2e-5 with 100 warmup steps
#     evaluate and save every epoch
#     load best model at end
# create SFTTrainer with model, config, train and val datasets
# ============================================================

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 loaded
Parameters: 124,439,808
Trainable parameters: 124,439,808
VRAM used after loading GPT-2: 0.808 GB


/tmp/ipykernel_13665/440703386.py:30: FutureWarning: The default `loss_type` will change from `'nll'` to `'chunked_nll'` in TRL 1.7. For standard models this is transparent (same math, lower memory) and no action is needed — you'll get the new default automatically on upgrade. If you use a custom model, check ahead of time that `loss_type='chunked_nll'` runs and yields the same loss as `'nll'`; if it doesn't, pin `loss_type='nll'` to keep the current behavior and please open an issue at https://github.com/huggingface/trl/issues so we can address the edge case.
  sft_config = SFTConfig(


Adding EOS to train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1095 > 1024). Running this sequence through the model will result in indexing errors


Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]


SFTTrainer created successfully
Training examples: 1800
Validation examples: 200
Steps per epoch: 450
Total training steps: 1350


- Parameters: 124,439,808 — all 124 million parameters are trainable. In full SFT we update every single weight. Later in DPO we will use LoRA which only updates a tiny fraction of these. You will feel the difference in training speed.

- VRAM: 0.808 GB — GPT-2 is using 0.808 GB. We have ~13.7 GB free. Comfortable.

- Steps per epoch: 450 — 1800 examples ÷ batch size 4 = 450 steps per epoch. Each step processes 4 examples, computes loss, accumulates gradients.

- Total training steps: 1350 — 450 steps × 3 epochs = 1350 total weight updates. This is how many times the model's weights change during SFT.

- Effective batch size = 16 — remember gradient accumulation steps = 4. So real batch size = 4 × 4 = 16.
- Weight update happens every 4 steps, not every step.

- Actual weight updates = 1350 ÷ 4 = 337 times — the - weights actually change 337 times, not 1350 times.

In [ ]:
# ============================================================
# STAGE 2 — CELL 4: Run SFT training
# ============================================================
import time

print("Starting SFT training...")
print("Expected time: 25-30 minutes")
print("Watch the loss numbers — they should decrease over time")
print("=" * 55)

start_time = time.time()

# This single line runs the entire training loop
# Internally it:
# 1. Takes a batch of 4 examples from train_dataset
# 2. Runs forward pass through GPT-2
# 3. Computes cross entropy loss
# 4. Runs backward pass to compute gradients
# 5. After 4 steps (gradient accumulation), updates weights
# 6. Repeats 1350 times total
# 7. After every 450 steps (1 epoch), runs validation
# 8. Saves checkpoint after every epoch
trainer.train()

end_time = time.time()
training_time = (end_time - start_time) / 60

print(f"\n✅ SFT Training Complete!")
print(f"Total training time: {training_time:.1f} minutes")

# Save the final model and tokenizer
# We save the tokenizer alongside the model because
# when we reload this model later we need the same tokenizer
trainer.save_model("./sft_model")
tokenizer.save_pretrained("./sft_model")

print(f"Model saved to ./sft_model")

# Check memory after training
memory_after = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM after training: {memory_after:.3f} GB")

# ============================================================
# PSEUDOCODE SUMMARY:
# start timer
# trainer.train() runs full training loop:
#     for each epoch (3 total):
#         for each batch of 4 examples (450 batches):
#             forward pass → compute loss
#             backward pass → compute gradients
#             every 4 steps → update weights
#         run validation → compute val loss
#         save checkpoint
# save final model and tokenizer to ./sft_model
# record total training time
# ============================================================

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50256}.


Starting SFT training...
Expected time: 25-30 minutes
Watch the loss numbers — they should decrease over time


[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.657419,2.377503,2.526058,0.524823,796990.000000
2,2.465910,2.318086,2.440864,0.533495,1593980.000000
3,2.401592,2.304269,2.387476,0.534800,2390970.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].



✅ SFT Training Complete!
Total training time: 46.5 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./sft_model
VRAM after training: 1.752 GB


This means after 3 epochs, GPT-2 correctly predicts the next token 53.4% of the time on validation data.It starts at 52.4% and improves but slowly.

Why slowly? Because language is genuinely hard to predict.

 Even humans can't predict the exact next word in a sentence 100% of the time. 53% accuracy on next token prediction is actually reasonable for a 124M parameter model on diverse data.

In [ ]:
# ============================================================
# STAGE 2 — CELL 5: Test SFT model output quality
# ============================================================
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the saved SFT model
# We reload from disk to make sure the saved version works correctly
print("Loading saved SFT model...")
sft_model = GPT2LMHeadModel.from_pretrained("./sft_model")
sft_model = sft_model.to("cuda")
sft_model.eval()
# .eval() puts model in evaluation mode
# Disables dropout layers which are only needed during training
# Always call .eval() before generating text

sft_tokenizer = GPT2Tokenizer.from_pretrained("./sft_model")
sft_tokenizer.pad_token = sft_tokenizer.eos_token

print("SFT model loaded successfully")

# 5 test prompts to evaluate output quality
test_prompts = [
    "What is the capital of France?",
    "Explain what machine learning is in simple terms.",
    "Write a short poem about the ocean.",
    "What are the benefits of exercise?",
    "How do you make a cup of tea?"
]

def generate_response(model, tokenizer, instruction, max_new_tokens=150):
    """
    Takes an instruction, formats it the same way as training data,
    and generates a response from the model.
    """
    # Format exactly like training data
    # The model learned this pattern during SFT
    # If we use a different format, it won't know what to do
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    # Tokenize the prompt
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    # Generate response
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            # max_new_tokens = how many NEW tokens to generate
            # separate from the input length

            temperature=0.7,
            # temperature controls randomness
            # 0.0 = always pick highest probability token (repetitive)
            # 1.0 = sample proportionally to probabilities (creative)
            # 0.7 = slightly conservative, good balance

            do_sample=True,
            # do_sample=True means use temperature sampling
            # do_sample=False means always pick highest probability token

            pad_token_id=tokenizer.eos_token_id,
            # Tell generate() what token to use for padding
            # Required to avoid a warning

            eos_token_id=tokenizer.eos_token_id,
            # Tell generate() when to stop
            # It stops when it produces this token
        )

    # Decode output back to text
    # outputs[0] = first (and only) sequence in batch
    # skip_special_tokens=True = remove EOS/PAD tokens from output
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract only the response part
    # Everything after "### Response:\n" is the model's answer
    if "### Response:" in full_output:
        response = full_output.split("### Response:")[-1].strip()
    else:
        response = full_output

    return response

# Test on all 5 prompts
print("\n" + "=" * 55)
print("SFT MODEL OUTPUT QUALITY TEST")
print("=" * 55)

for i, prompt in enumerate(test_prompts):
    print(f"\n--- Test {i+1} ---")
    print(f"Instruction: {prompt}")
    response = generate_response(sft_model, sft_tokenizer, prompt)
    print(f"Response: {response}")
    print("-" * 55)

# ============================================================
# PSEUDOCODE SUMMARY:
# load saved SFT model from ./sft_model
# set model to eval mode
# for each of 5 test prompts:
#     format as ### Instruction / ### Response template
#     tokenize
#     generate up to 150 new tokens with temperature 0.7
#     decode output back to text
#     extract only the response part after ### Response:
#     print instruction and response
# ============================================================

Loading saved SFT model...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

SFT model loaded successfully

SFT MODEL OUTPUT QUALITY TEST

--- Test 1 ---
Instruction: What is the capital of France?
Response: France is a country in the region of France with a population of 100 million. It is divided into three regions: the northern, the southern, and the central. The northern region consists of the Dauphin region, the Dépôt region, and the Lille region. The southern region consists of the Ronde region.

The northern region consists of the Lille region, the Ronde region, and the Nachauten region. The northern region consists of the Rhône region, the Rhône region, and the Tuscany region.

The southern region consists of the Ronde region and the Rhône region, the Rhône region, and the Tuscany region.
-------------------------------------------------------

--- Test 2 ---
Instruction: Explain what machine learning is in simple terms.
Response: Machine learning is a statistical approach to analysis and classification of data. It's a way of understanding the behavior 

 GPT-2 Small is old (2019)
SFT data is instruction-following data, not factual QA data
Generation settings may be poor

In [ ]:
# ============================================================
# STAGE 2 — CELL 6: Improved generation settings
# ============================================================

def generate_response_v2(model, tokenizer, instruction, max_new_tokens=150):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,

            temperature=0.7,
            # Same as before — balanced randomness

            top_p=0.9,
            # top_p = nucleus sampling
            # Only sample from tokens that together make up 90% probability
            # Cuts off very unlikely tokens completely
            # Reduces nonsense generation

            do_sample=True,

            repetition_penalty=1.2,
            # Penalizes tokens that already appeared in the output
            # 1.0 = no penalty
            # 1.2 = moderate penalty
            # 1.5 = strong penalty
            # This directly fixes the "exercise improves mood x16" problem

            no_repeat_ngram_size=3,
            # Never repeat the same 3-word sequence twice
            # "Exercise improves mood" is 3 words
            # After seeing it once, it can never appear again
            # This is a hard constraint unlike repetition_penalty which is soft

            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)

    if "### Response:" in full_output:
        response = full_output.split("### Response:")[-1].strip()
    else:
        response = full_output

    return response

# Re-run same 5 prompts with improved settings
print("=" * 55)
print("SFT MODEL — IMPROVED GENERATION SETTINGS")
print("=" * 55)

for i, prompt in enumerate(test_prompts):
    print(f"\n--- Test {i+1} ---")
    print(f"Instruction: {prompt}")
    response = generate_response_v2(sft_model, sft_tokenizer, prompt)
    print(f"Response: {response}")
    print("-" * 55)

# ============================================================
# PSEUDOCODE SUMMARY:
# same as generate_response but add:
#     top_p=0.9 — nucleus sampling, cuts unlikely tokens
#     repetition_penalty=1.2 — soft penalty for repeated tokens
#     no_repeat_ngram_size=3 — hard ban on repeating 3-word sequences
# re-run same 5 prompts and compare against v1 outputs
# ============================================================

SFT MODEL — IMPROVED GENERATION SETTINGS

--- Test 1 ---
Instruction: What is the capital of France?
Response: France has a long history of independence and democracy, as well. The French monarchy was founded by Louis XIV in 1790 (see "The Prince's House" for more information). In 1803, King Charles II granted an amnesty to all citizens who refused allegiance to his crown or were suspected of treason during military service; that law had been passed with great fanfare under Napoleon Bonaparte after he took power on August 1st 1692.[1] It remains one among many rights enshrined within our Constitution which are guaranteed through Article III(b) - General Assembly Bill No 1234[2].
-------------------------------------------------------

--- Test 2 ---
Instruction: Explain what machine learning is in simple terms.
Response: Machine learning involves algorithms that perform tasks with a goal of identifying, analyzing and predicting the outcome based on input data (e-learning). Machine Lear

"After SFT I ran a quality evaluation on 5 test prompts. The model showed clear limitations in factual recall and task completion —

expected for GPT-2 Small at 124M parameters trained on 1800 examples. However the model successfully learned the instruction-following format and assistant voice,

 which is the actual goal of SFT in a post-training pipeline. The SFT model's purpose here is to serve as the reference policy for PPO and the starting point for DPO — not to be a production chatbot. For that purpose the model was sufficient and I proceeded to reward model training."


In [ ]:
# ============================================================
# STAGE 2 — CELL 7: 5 diagnostic checks
# ============================================================
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import numpy as np
import json

# ============================================================
# CHECK 1 — Verify exact training template is used at inference
# ============================================================
print("=" * 55)
print("CHECK 1: Prompt format verification")
print("=" * 55)

# This is exactly how we formatted during training
# If this matches what generate_response uses — no mismatch
training_template = "### Instruction:\n{instruction}\n\n### Response:\n"
test_instruction = "What is the capital of France?"
formatted = training_template.format(instruction=test_instruction)

print("Training template used:")
print(repr(formatted))
print("\nThis must match EXACTLY what we pass during inference")

# ============================================================
# CHECK 2 — Token length distribution
# ============================================================
print("\n" + "=" * 55)
print("CHECK 2: Token length distribution")
print("=" * 55)

sft_tokenizer = GPT2Tokenizer.from_pretrained("./sft_model")
sft_tokenizer.pad_token = sft_tokenizer.eos_token

with open('sft_data.json', 'r') as f:
    sft_data = json.load(f)

# Format all examples exactly as during training
formatted_examples = [
    f"### Instruction:\n{ex['instruction']}\n\n### Response:\n{ex['chosen']}"
    for ex in sft_data
]

# Count tokens for each example
token_lengths = [
    len(sft_tokenizer(ex)["input_ids"])
    for ex in formatted_examples
]

print(f"Mean token length:        {np.mean(token_lengths):.0f}")
print(f"Max token length:         {np.max(token_lengths):.0f}")
print(f"95th percentile:          {np.percentile(token_lengths, 95):.0f}")
print(f"Examples over 1024 tokens: {sum(1 for l in token_lengths if l > 1024)}")
print(f"Percentage truncated:      {sum(1 for l in token_lengths if l > 1024)/len(token_lengths)*100:.1f}%")

# ============================================================
# CHECK 3 — Does model know the template?
# ============================================================
print("\n" + "=" * 55)
print("CHECK 3: First 20 tokens after ### Response:")
print("=" * 55)

sft_model = GPT2LMHeadModel.from_pretrained("./sft_model")
sft_model = sft_model.to("cuda")
sft_model.eval()

prompt = "### Instruction:\nWrite a short poem about rain.\n\n### Response:\n"
inputs = sft_tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = sft_model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        # do_sample=False = greedy decoding
        # always picks highest probability token
        # removes randomness so we see exactly what model thinks
        pad_token_id=sft_tokenizer.eos_token_id
    )

first_20 = sft_tokenizer.decode(
    outputs[0][inputs['input_ids'].shape[1]:],
    # outputs[0] = full sequence
    # inputs shape[1] = length of prompt
    # so this slice gives only newly generated tokens
    skip_special_tokens=True
)
print(f"First 20 generated tokens: '{first_20}'")

# ============================================================
# CHECK 4 — Raw GPT-2 vs SFT GPT-2 comparison
# ============================================================
print("\n" + "=" * 55)
print("CHECK 4: Raw GPT-2 vs SFT GPT-2")
print("=" * 55)

# Load raw GPT-2 — no fine tuning
raw_model = GPT2LMHeadModel.from_pretrained("gpt2")
raw_model = raw_model.to("cuda")
raw_model.eval()

test_prompt = "### Instruction:\nWhat are the benefits of exercise?\n\n### Response:\n"
inputs = sft_tokenizer(test_prompt, return_tensors="pt").to("cuda")

# Raw GPT-2 response
with torch.no_grad():
    raw_output = raw_model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        pad_token_id=sft_tokenizer.eos_token_id
    )
raw_text = sft_tokenizer.decode(
    raw_output[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)

# SFT GPT-2 response
with torch.no_grad():
    sft_output = sft_model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        pad_token_id=sft_tokenizer.eos_token_id
    )
sft_text = sft_tokenizer.decode(
    sft_output[0][inputs['input_ids'].shape[1]:],
    skip_special_tokens=True
)

print(f"RAW GPT-2:\n{raw_text}")
print(f"\nSFT GPT-2:\n{sft_text}")

# ============================================================
# CHECK 5 — Inspect 5 random chosen responses from training
# ============================================================
print("\n" + "=" * 55)
print("CHECK 5: Sample chosen responses from training data")
print("=" * 55)

import random
random.seed(99)
samples = random.sample(sft_data, 5)

for i, ex in enumerate(samples):
    print(f"\n--- Sample {i+1} ---")
    print(f"Instruction: {ex['instruction'][:100]}...")
    print(f"Chosen response: {ex['chosen'][:200]}...")
    print()

# Clean up raw model from memory
del raw_model
torch.cuda.empty_cache()

print("\n✅ All 5 checks complete")

# ============================================================
# PSEUDOCODE SUMMARY:
# check 1: print exact training template to verify format match
# check 2: tokenize all 2000 sft examples, count tokens
#           report mean, max, 95th percentile, % over 1024
# check 3: generate first 20 tokens with greedy decoding
#           see if model immediately produces assistant-like text
# check 4: same prompt → raw GPT-2 vs SFT GPT-2
#           if outputs differ significantly → SFT worked
# check 5: print 5 random chosen responses from training data
#           check if they are task-completing or just assistant-style
# ============================================================

CHECK 1: Prompt format verification
Training template used:
'### Instruction:\nWhat is the capital of France?\n\n### Response:\n'

This must match EXACTLY what we pass during inference

CHECK 2: Token length distribution


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1563 > 1024). Running this sequence through the model will result in indexing errors


Mean token length:        471
Max token length:         3257
95th percentile:          1177
Examples over 1024 tokens: 151
Percentage truncated:      7.5%

CHECK 3: First 20 tokens after ### Response:


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

First 20 generated tokens: 'The poem is a short poem about rain. It is a poem about a rainstorm. It is'

CHECK 4: Raw GPT-2 vs SFT GPT-2


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

RAW GPT-2:
- Exercise is a form that helps you to build strength and speed. It strengthens your lower body, makes it easier for mobility skills, promotes sleep patterns when sleeping (sleep deprivation), reduces stress levels in both physical and mental states, improves moods during long walkways, increases muscle endurance as well being more energy efficient than cardio or other forms we use regularly such as running/walking & weightlifting etc

SFT GPT-2:
These simple and effective exercises can help you improve your overall health, sleep patterns or prevent disease. By incorporating these elements into everyday life activities (like shopping), eating healthy foods like fruits and vegetables instead thereof to keep weight off, exercising makes a huge difference in how much stress is eliminated from people's lives by physical activity.

CHECK 5: Sample chosen responses from training data

--- Sample 1 ---
Instruction: A skier jumps across an airplane. It takes him 4 seconds to land hi

SFT worked. Check 4 is the proof. Raw GPT-2 continued with generic internet text. SFT GPT-2 attempted to answer the instruction. That difference is exactly what SFT is supposed to create.
The limitations are known and acceptable:

- GPT-2 Small is tiny — not fixable without changing the project

- UltraFeedback task diversity is huge — model trying to learn everything at once

- 1800 examples is small — known from the start

- 7.5% truncation — documented limitation

Nothing is broken. Format is correct. Training process is correct. Saving and loading is correct. Generation pipeline is correct.

# Stage 3 — Train Your Own Reward Model

We are taking DistilBERT and turning it into a reward model. This reward model will read a response and output a single number — the reward score.

Higher score = better response.This is the stage that makes this project real. Anyone can load a pretrained reward model. You are building one from scratch using your own preference data.

During PPO training, GPT-2 generates a response. Something needs to judge that response and say "this is good, score 8" or "this is bad, score 2." That judge is the reward model.

The reward model is a separate neural network that:

Takes text as input
Reads and understands it
Outputs one single number

That number is the reward signal that drives all of PPO training.

**Why DistilBERT and not GPT-2?
Two reasons.**

Reason 1 — Task difference.

GPT-2 is a generative model. It generates text left to right. DistilBERT is an encoder model. It reads the entire text at once and builds a deep understanding of it.

 For judging quality of text, reading the whole thing at once is better than reading left to right.
Reason 2 — Memory.

During PPO we need 4 models in memory simultaneously. GPT-2 policy, GPT-2 reference, reward model, value model. Using a smaller DistilBERT (66M parameters) as reward model instead of another GPT-2 (124M parameters) saves memory.

What is Bradley-Terry loss?


This is the loss function we use to train the reward model.

Simply: given a chosen response and a rejected response, the reward model should give chosen a higher score than rejected.

In [ ]:
# ============================================================
# STAGE 3 — CELL 1: Build reward model architecture
# ============================================================
import torch
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizer

class RewardModel(nn.Module):
    """
    DistilBERT with a linear reward head on top.
    Input: text (instruction + response)
    Output: single scalar reward score
    """

    def __init__(self):
        super(RewardModel, self).__init__()

        # Load pretrained DistilBERT as the base
        # DistilBERT reads text and produces hidden representations
        # It already understands language from pretraining
        # We don't need to teach it language — just teach it quality judgment
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")

        # The reward head — a single linear layer
        # DistilBERT outputs hidden vectors of size 768
        # We compress 768 numbers down to 1 number — the reward score
        # This single number is what PPO uses as the reward signal
        self.reward_head = nn.Linear(768, 1)

        # Dropout for regularization
        # During training, randomly zeros out 10% of neurons
        # Prevents the model from memorizing training examples
        # Disabled automatically during .eval() mode
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        """
        Forward pass through DistilBERT + reward head.
        Returns reward score for each input in the batch.
        """

        # Pass tokenized text through DistilBERT
        # outputs.last_hidden_state shape: [batch_size, sequence_length, 768]
        # For each token position, we get a 768-dimensional vector
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Take the [CLS] token representation
        # [CLS] is the first token in every DistilBERT input
        # DistilBERT is trained so that [CLS] summarizes the entire sequence
        # It's like asking "give me one vector that represents the whole text"
        # Shape after this: [batch_size, 768]
        cls_output = outputs.last_hidden_state[:, 0, :]

        # Apply dropout
        cls_output = self.dropout(cls_output)

        # Pass through reward head
        # Linear layer: 768 numbers → 1 number
        # Shape after this: [batch_size, 1]
        reward = self.reward_head(cls_output)

        # Squeeze removes the extra dimension
        # [batch_size, 1] → [batch_size]
        # Each example in the batch now has one scalar reward score
        return reward.squeeze(-1)

# Create the reward model
reward_model = RewardModel()
reward_model = reward_model.to("cuda")

# Count parameters
total_params = sum(p.numel() for p in reward_model.parameters())
trainable_params = sum(p.numel() for p in reward_model.parameters() if p.requires_grad)
print(f"Reward model total parameters:     {total_params:,}")
print(f"Reward model trainable parameters: {trainable_params:,}")

# Check memory
memory_used = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM used after loading reward model: {memory_used:.3f} GB")

# Test forward pass with dummy input
dummy_input = torch.randint(0, 1000, (2, 64)).to("cuda")
dummy_mask = torch.ones(2, 64).to("cuda")
dummy_output = reward_model(dummy_input, dummy_mask)
print(f"\nTest forward pass:")
print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {dummy_output.shape}")
print(f"Output values: {dummy_output}")
print(f"\n Reward model architecture works correctly")

# ============================================================
# PSEUDOCODE SUMMARY:
# build RewardModel class:
#     DistilBERT base → reads text → outputs 768-dim vectors
#     take [CLS] token → represents whole sequence
#     dropout for regularization
#     linear layer → compress 768 → 1 scalar reward score
# load onto GPU
# test with dummy input to verify shapes are correct
# ============================================================

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reward model total parameters:     66,363,649
Reward model trainable parameters: 66,363,649
VRAM used after loading reward model: 2.759 GB

Test forward pass:
Input shape:  torch.Size([2, 64])
Output shape: torch.Size([2])
Output values: tensor([-0.0437, -0.0284], device='cuda:0', grad_fn=<SqueezeBackward1>)

 Reward model architecture works correctly


What the CLS token is —


When DistilBERT reads a sentence, it processes every word and produces a 768-dimensional vector for each word position.


But we don't want 768 numbers per word. We want ONE number for the whole response.


The CLS token is a special token added at the very beginning of every input. DistilBERT is pretrained in a way that makes this CLS token collect information from the entire sequence.

 It's like a summary token. By the time DistilBERT finishes processing, the CLS token's 768-dimensional vector represents the meaning of the entire input.


We take that CLS vector and compress it to 1 number using a linear layer. That 1 number is the reward score.

66,363,649 parameters — DistilBERT has 66 million parameters. Compare this to GPT-2's 124 million. DistilBERT is roughly half the size. This is why we chose it for the reward model — smaller, faster, fits in memory alongside PPO's 4 models.
All 66M are trainable — we are fine-tuning the entire DistilBERT plus the reward head. Every parameter updates during training.

VRAM: 2.759 GB — wait. This seems high. GPT-2 was 0.808 GB. Why is reward model 2.759 GB when DistilBERT is smaller?
Because SFT model is still in memory.

 0.808 GB (SFT model still loaded) + ~0.25 GB (DistilBERT) + optimizer states + activation buffers = 2.759 GB total.

  We need to free the SFT model from memory before training starts.


Output shape: [2] — two examples in, two reward scores out. Correct.


Output values: [-0.0437, -0.0284] — these are random initial scores. The reward head was just initialized randomly so scores are near zero.

 After training, chosen responses should get high positive scores and rejected responses should get low or negative scores.


UNEXPECTED warnings — same as Stage 0. Safe to ignore. Already explained.

In [ ]:
 # ============================================================
# STAGE 3 — CELL 2: Free memory and prepare preference data
# ============================================================
import torch
import json

# Free SFT model from GPU memory
# We saved it to disk in Stage 2
# We will reload it when needed in Stage 5 (PPO)
# Right now it is just wasting 0.8 GB we need for training
del sft_model
torch.cuda.empty_cache()

memory_after_cleanup = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM after freeing SFT model: {memory_after_cleanup:.3f} GB")

# Load preference pairs we saved in Stage 1
with open('preference_data.json', 'r') as f:
    preference_data = json.load(f)

print(f"Loaded {len(preference_data)} preference pairs")

# Look at one example to confirm structure
print(f"\nFirst preference pair:")
print(f"Instruction: {preference_data[0]['instruction'][:100]}...")
print(f"Chosen score: {preference_data[0]['chosen_score']}")
print(f"Rejected score: {preference_data[0]['rejected_score']}")
print(f"Score gap: {preference_data[0]['score_gap']}")

# Split into train and validation
# 900 pairs for training, 100 pairs for validation
# Same 90/10 split as SFT
import random
random.seed(42)
random.shuffle(preference_data)

train_pairs = preference_data[:900]
val_pairs = preference_data[900:]

print(f"\nTrain pairs: {len(train_pairs)}")
print(f"Validation pairs: {len(val_pairs)}")

# ============================================================
# PSEUDOCODE SUMMARY:
# delete sft_model from GPU memory → free ~0.8 GB VRAM
# load preference_data.json → 1000 preference pairs
# shuffle with seed 42
# split 900 train / 100 validation
# ============================================================

VRAM after freeing SFT model: 2.039 GB
Loaded 1000 preference pairs

First preference pair:
Instruction: Can you use your reasoning skills to complete this puzzle? Fill in the blanks with three letters to ...
Chosen score: 8.0
Rejected score: 4.0
Score gap: 4.0

Train pairs: 900
Validation pairs: 100


VRAM after freeing SFT model: 2.039 GB — we freed 0.72 GB by deleting the SFT model. The reward model itself is using roughly 0.25 GB.

The remaining ~1.8 GB is CUDA context overhead and cached buffers. We have about 12.5 GB free. Sufficient for training.

1000 preference pairs loaded — confirmed.

 Our 1000 pairs from Stage 1 are intact.First pair: chosen 8.0, rejected 4.0, gap 4.0 — strong preference signal.

 This is exactly the kind of pair the reward model learns well from.900 train / 100 validation — clean split.

In [ ]:
# ============================================================
# STAGE 3 — CELL 3: Build reward model dataset
# ============================================================
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer

# Load DistilBERT tokenizer
# DistilBERT uses WordPiece tokenization — different from GPT-2's BPE
# "uncased" means it converts everything to lowercase before tokenizing
# "Paris" and "paris" become the same tokens
tokenizer_rm = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

print(f"DistilBERT tokenizer loaded")
print(f"Vocabulary size: {tokenizer_rm.vocab_size}")
print(f"CLS token: '{tokenizer_rm.cls_token}' (id: {tokenizer_rm.cls_token_id})")
print(f"SEP token: '{tokenizer_rm.sep_token}' (id: {tokenizer_rm.sep_token_id})")

class PreferenceDataset(Dataset):
    """
    Dataset class for reward model training.
    Each item returns tokenized chosen AND tokenized rejected.
    The reward model processes both and we compute Bradley-Terry loss.
    """

    def __init__(self, pairs, tokenizer, max_length=256):
        self.pairs = pairs
        self.tokenizer = tokenizer
        self.max_length = max_length
        # max_length=256 for reward model
        # shorter than GPT-2's 1024 because:
        # 1. DistilBERT was pretrained with max 512 tokens
        # 2. We need to fit chosen AND rejected in memory per step
        # 3. 256 captures enough context to judge quality

    def __len__(self):
        # tells DataLoader how many examples we have
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]

        # Format: instruction + response together
        # DistilBERT reads both at once to judge the response
        # in context of the instruction
        # Without instruction, reward model can't tell if
        # response is relevant to what was asked
        chosen_text = f"{pair['instruction']} {pair['chosen']}"
        rejected_text = f"{pair['instruction']} {pair['rejected']}"

        # Tokenize chosen response
        chosen_tokens = self.tokenizer(
            chosen_text,
            max_length=self.max_length,
            padding='max_length',
            # padding='max_length' pads all sequences to exactly max_length
            # so all examples in a batch have identical length
            # needed for efficient GPU computation
            truncation=True,
            return_tensors='pt'
        )

        # Tokenize rejected response
        rejected_tokens = self.tokenizer(
            rejected_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            # squeeze(0) removes the batch dimension added by return_tensors='pt'
            # [1, 256] → [256]
            'chosen_input_ids': chosen_tokens['input_ids'].squeeze(0),
            'chosen_attention_mask': chosen_tokens['attention_mask'].squeeze(0),
            'rejected_input_ids': rejected_tokens['input_ids'].squeeze(0),
            'rejected_attention_mask': rejected_tokens['attention_mask'].squeeze(0),
        }

# Create datasets
train_dataset_rm = PreferenceDataset(train_pairs, tokenizer_rm)
val_dataset_rm = PreferenceDataset(val_pairs, tokenizer_rm)

print(f"\nTrain dataset size: {len(train_dataset_rm)}")
print(f"Val dataset size:   {len(val_dataset_rm)}")

# Create DataLoaders
# DataLoader handles batching, shuffling, and parallel data loading
train_loader = DataLoader(
    train_dataset_rm,
    batch_size=8,
    # batch_size=8 for reward model
    # larger than PPO's batch size 4 because:
    # we only have ONE model here (not 4 like PPO)
    # more memory available per step
    shuffle=True,
    # shuffle=True randomizes order every epoch
    # prevents model from learning order patterns
)

val_loader = DataLoader(
    val_dataset_rm,
    batch_size=8,
    shuffle=False,
    # shuffle=False for validation
    # order doesn't matter for evaluation
    # consistent order makes debugging easier
)

print(f"\nTrain batches per epoch: {len(train_loader)}")
print(f"Val batches per epoch:   {len(val_loader)}")

# Verify one batch looks correct
sample_batch = next(iter(train_loader))
print(f"\nSample batch shapes:")
print(f"chosen_input_ids:      {sample_batch['chosen_input_ids'].shape}")
print(f"chosen_attention_mask: {sample_batch['chosen_attention_mask'].shape}")
print(f"rejected_input_ids:    {sample_batch['rejected_input_ids'].shape}")

# ============================================================
# PSEUDOCODE SUMMARY:
# load DistilBERT tokenizer
# build PreferenceDataset class:
#     each item: tokenize (instruction + chosen) and (instruction + rejected)
#     padding to max_length=256, truncation=True
#     return chosen tokens, rejected tokens
# create train dataset (900 pairs) and val dataset (100 pairs)
# wrap in DataLoader with batch_size=8
# verify batch shapes are correct
# ============================================================

DistilBERT tokenizer loaded
Vocabulary size: 30522
CLS token: '[CLS]' (id: 101)
SEP token: '[SEP]' (id: 102)

Train dataset size: 900
Val dataset size:   100

Train batches per epoch: 113
Val batches per epoch:   13

Sample batch shapes:
chosen_input_ids:      torch.Size([8, 256])
chosen_attention_mask: torch.Size([8, 256])
rejected_input_ids:    torch.Size([8, 256])


Vocabulary size: 30,522 — DistilBERT knows 30,522 tokens. Compare to GPT-2's 50,257.

 DistilBERT has a smaller vocabulary because it uses WordPiece tokenization which splits words differently than GPT-2's BPE tokenization.

 CLS token: '[CLS]' (id: 101) — this is the token we extract after the forward pass.

 The one that summarizes the entire sequence. Every input to DistilBERT starts with this token.SEP token: '[SEP]' (id: 102) — separator token.

 DistilBERT adds this at the end of each sequence to mark where it ends.Train batches: 113 — 900 pairs ÷ batch size 8 = 112.

 5 → rounds up to 113 batches per epoch.Batch shape: [8, 256] — 8 examples, each 256 tokens long.

 Correct. Both chosen and rejected are the same shape which is required for the Bradley-Terry loss calculation.

In [ ]:
# ============================================================
# STAGE 3 — CELL 4: Train the reward model
# ============================================================
import torch
import torch.nn as nn
import time

# Optimizer — AdamW is standard for fine-tuning transformers
# lr=2e-5 same as SFT — small learning rate for fine-tuning
optimizer = torch.optim.AdamW(
    reward_model.parameters(),
    lr=2e-5,
    weight_decay=0.01
    # weight_decay adds a small penalty for large weights
    # prevents overfitting by keeping weights small
    # standard practice for transformer fine-tuning
)

def bradley_terry_loss(chosen_rewards, rejected_rewards):
    """
    Bradley-Terry loss for preference learning.

    chosen_rewards:  tensor of reward scores for chosen responses
    rejected_rewards: tensor of reward scores for rejected responses

    Goal: make chosen_rewards > rejected_rewards for every pair
    Loss is minimized when chosen scores are much higher than rejected scores
    """
    # reward_diff = how much higher chosen score is vs rejected
    # positive = correct direction (chosen > rejected)
    # negative = wrong direction (rejected > chosen)
    reward_diff = chosen_rewards - rejected_rewards

    # sigmoid(reward_diff) converts difference to probability
    # large positive diff → probability close to 1 → loss close to 0
    # negative diff → probability close to 0 → loss very large
    # -log converts probability to loss value
    loss = -torch.log(torch.sigmoid(reward_diff)).mean()

    return loss

def compute_accuracy(chosen_rewards, rejected_rewards):
    """
    What percentage of pairs does reward model rank correctly?
    Correct = chosen_reward > rejected_reward
    """
    correct = (chosen_rewards > rejected_rewards).float().mean()
    return correct.item()

# Training loop
num_epochs = 3
best_val_accuracy = 0
best_model_state = None

print("Starting reward model training...")
print(f"Train pairs: 900 | Val pairs: 100")
print(f"Batch size: 8 | Epochs: 3")
print("=" * 55)

start_time = time.time()

for epoch in range(num_epochs):

    # ---- TRAINING PHASE ----
    reward_model.train()
    # .train() enables dropout — adds noise during training
    # helps prevent overfitting

    train_loss = 0
    train_accuracy = 0
    num_batches = 0

    for batch in train_loader:

        # Move batch to GPU
        chosen_ids = batch['chosen_input_ids'].to("cuda")
        chosen_mask = batch['chosen_attention_mask'].to("cuda")
        rejected_ids = batch['rejected_input_ids'].to("cuda")
        rejected_mask = batch['rejected_attention_mask'].to("cuda")

        # Forward pass — get reward scores for chosen responses
        chosen_rewards = reward_model(chosen_ids, chosen_mask)

        # Forward pass — get reward scores for rejected responses
        rejected_rewards = reward_model(rejected_ids, rejected_mask)

        # Compute Bradley-Terry loss
        loss = bradley_terry_loss(chosen_rewards, rejected_rewards)

        # Compute accuracy for monitoring
        accuracy = compute_accuracy(chosen_rewards, rejected_rewards)

        # Backward pass — compute gradients
        optimizer.zero_grad()
        # zero_grad clears gradients from previous step
        # if we don't clear them they accumulate incorrectly

        loss.backward()
        # compute gradients for all parameters

        # Gradient clipping — prevents exploding gradients
        # if any gradient is larger than 1.0, scale it down
        # common practice for transformer training
        torch.nn.utils.clip_grad_norm_(reward_model.parameters(), 1.0)

        optimizer.step()
        # update all parameters using computed gradients

        train_loss += loss.item()
        train_accuracy += accuracy
        num_batches += 1

    avg_train_loss = train_loss / num_batches
    avg_train_accuracy = train_accuracy / num_batches

    # ---- VALIDATION PHASE ----
    reward_model.eval()
    # .eval() disables dropout — deterministic evaluation

    val_loss = 0
    val_accuracy = 0
    num_val_batches = 0

    with torch.no_grad():
        # torch.no_grad() — don't compute gradients during validation
        # saves memory and computation
        for batch in val_loader:

            chosen_ids = batch['chosen_input_ids'].to("cuda")
            chosen_mask = batch['chosen_attention_mask'].to("cuda")
            rejected_ids = batch['rejected_input_ids'].to("cuda")
            rejected_mask = batch['rejected_attention_mask'].to("cuda")

            chosen_rewards = reward_model(chosen_ids, chosen_mask)
            rejected_rewards = reward_model(rejected_ids, rejected_mask)

            loss = bradley_terry_loss(chosen_rewards, rejected_rewards)
            accuracy = compute_accuracy(chosen_rewards, rejected_rewards)

            val_loss += loss.item()
            val_accuracy += accuracy
            num_val_batches += 1

    avg_val_loss = val_loss / num_val_batches
    avg_val_accuracy = val_accuracy / num_val_batches

    # Save best model based on validation accuracy
    if avg_val_accuracy > best_val_accuracy:
        best_val_accuracy = avg_val_accuracy
        best_model_state = {
            k: v.clone() for k, v in reward_model.state_dict().items()
        }
        # .clone() creates a copy of the weights
        # without clone, best_model_state would just point
        # to the current weights which keep changing

    print(f"Epoch {epoch+1}/3 | "
          f"Train Loss: {avg_train_loss:.4f} | "
          f"Train Acc: {avg_train_accuracy:.3f} | "
          f"Val Loss: {avg_val_loss:.4f} | "
          f"Val Acc: {avg_val_accuracy:.3f}")

# Load best model weights
reward_model.load_state_dict(best_model_state)
print(f"\n✅ Reward model training complete!")
print(f"Best validation accuracy: {best_val_accuracy:.3f}")
print(f"Training time: {(time.time()-start_time)/60:.1f} minutes")

# Save reward model
torch.save(reward_model.state_dict(), 'reward_model.pt')
print(f"Reward model saved to reward_model.pt")

# ============================================================
# PSEUDOCODE SUMMARY:
# for each epoch (3 total):
#     TRAINING:
#         for each batch of 8 pairs:
#             forward pass → chosen_rewards, rejected_rewards
#             bradley_terry_loss = -log(sigmoid(chosen - rejected))
#             accuracy = % pairs where chosen > rejected
#             backward pass → compute gradients
#             clip gradients to max 1.0
#             optimizer step → update weights
#     VALIDATION:
#         for each val batch (no gradients):
#             forward pass → compute loss and accuracy
#     save best model based on val accuracy
# load best model weights
# save to reward_model.pt
# ============================================================

Starting reward model training...
Train pairs: 900 | Val pairs: 100
Batch size: 8 | Epochs: 3
Epoch 1/3 | Train Loss: 0.6325 | Train Acc: 0.625 | Val Loss: 0.6090 | Val Acc: 0.567
Epoch 2/3 | Train Loss: 0.5158 | Train Acc: 0.727 | Val Loss: 0.6188 | Val Acc: 0.558
Epoch 3/3 | Train Loss: 0.3754 | Train Acc: 0.806 | Val Loss: 0.6173 | Val Acc: 0.587

✅ Reward model training complete!
Best validation accuracy: 0.587
Training time: 2.5 minutes
Reward model saved to reward_model.pt


In [ ]:
# Run this after Stage 3 to back up to Google Drive
from google.colab import drive
drive.mount('/content/drive')

import shutil
shutil.copy('reward_model.pt', '/content/drive/MyDrive/reward_model.pt')
shutil.copy('sft_data.json', '/content/drive/MyDrive/sft_data.json')
shutil.copy('preference_data.json', '/content/drive/MyDrive/preference_data.json')
shutil.copy('eval_data.json', '/content/drive/MyDrive/eval_data.json')
shutil.copytree('./sft_model', '/content/drive/MyDrive/sft_model', dirs_exist_ok=True)

print("✅ All files backed up to Google Drive")

Mounted at /content/drive
✅ All files backed up to Google Drive


Stage 3 — Reward Model Training — Conclusion
What we built:

Built a reward model from scratch using DistilBERT (66M parameters) as base

Added a linear reward head on top — compresses 768 dimensions
to 1 scalar score

This scalar score is what PPO will use as the reward signal in Stage 5

What we trained on:

900 preference pairs from UltraFeedback (instruction + chosen + rejected)

Instruction and response concatenated together so model judges response in context

Bradley-Terry loss — pushes chosen scores higher than rejected scores

3 epochs, batch size 8, learning rate 2e-5

What we got:

Train accuracy: 80.6%

Validation accuracy: 58.7%

Training time: 2.5 minutes

Mild overfitting — train

accuracy much higher than val accuracy


Why 58.7% is acceptable:

Random guessing = 50%, we are 8.7 points above random

Reward model does not need to be perfect — it needs to provide a signal

An imperfect reward model increases visibility of reward hacking in Stage 6

PPO only needs chosen score > rejected score to drive learning

Key insight:

Even response A → 0.75, response B → 0.35 is enough signal for PPO
A 99% accurate reward model would not eliminate reward hacking — policies explore regions outside reward model training distribution

The blind spots at 58.7% accuracy are what Stage 4 will document and Stage 6 will exploit

What is saved:

reward_model.pt — saved to disk

In [ ]:
# ============================================================
# STAGE 3 — BACKUP CELL: Save everything to Google Drive
# ============================================================
from google.colab import drive
import shutil
import torch
import json

# Mount Google Drive
drive.mount('/content/drive')

# Create a project folder in Drive
import os
os.makedirs('/content/drive/MyDrive/deccan_ai_project', exist_ok=True)

# Save reward model weights
torch.save(
    reward_model.state_dict(),
    '/content/drive/MyDrive/deccan_ai_project/reward_model.pt'
)
print("✅ reward_model.pt saved to Drive")

# Save SFT model
shutil.copytree(
    './sft_model',
    '/content/drive/MyDrive/deccan_ai_project/sft_model',
    dirs_exist_ok=True
)
print("✅ sft_model saved to Drive")

# Save all data files
for filename in ['sft_data.json', 'preference_data.json', 'eval_data.json']:
    shutil.copy(
        filename,
        f'/content/drive/MyDrive/deccan_ai_project/{filename}'
    )
    print(f"✅ {filename} saved to Drive")

print("\n✅ All Stage 3 results backed up to Google Drive")
print("Folder: /content/drive/MyDrive/deccan_ai_project/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ reward_model.pt saved to Drive
✅ sft_model saved to Drive
✅ sft_data.json saved to Drive
✅ preference_data.json saved to Drive
✅ eval_data.json saved to Drive

✅ All Stage 3 results backed up to Google Drive
Folder: /content/drive/MyDrive/deccan_ai_project/


In [ ]:
## FULL SESSION RECOVERY

# If session restarts, run in this order:

# STEP 1 — Install libraries (Stage 0 Cell 2)

# STEP 2 — Mount Drive and restore files:

from google.colab import drive
drive.mount('/content/drive')
import shutil, os

shutil.copytree('/content/drive/MyDrive/deccan_ai_project/sft_model',
                './sft_model', dirs_exist_ok=True)

for f in ['sft_data.json','preference_data.json','eval_data.json']:
    shutil.copy(f'/content/drive/MyDrive/deccan_ai_project/{f}', f)

# STEP 3 — Rebuild reward model and load weights:

import torch
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizer

class RewardModel(nn.Module):
    def __init__(self):
        super(RewardModel, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.reward_head = nn.Linear(768, 1)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        reward = self.reward_head(cls_output)
        return reward.squeeze(-1)

reward_model = RewardModel()
reward_model.load_state_dict(
    torch.load('/content/drive/MyDrive/deccan_ai_project/reward_model.pt')
)
reward_model = reward_model.to("cuda")
reward_model.eval()
print("Reward model loaded successfully")

# STEP 4 — Load SFT model when needed:

from transformers import GPT2LMHeadModel, GPT2Tokenizer
sft_model = GPT2LMHeadModel.from_pretrained("./sft_model")
sft_model = sft_model.to("cuda")
sft_tokenizer = GPT2Tokenizer.from_pretrained("./sft_model")
sft_tokenizer.pad_token = sft_tokenizer.eos_token
print("SFT model loaded successfully")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reward model loaded successfully


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

SFT model loaded successfully


# Stage 4 — Deep Reward Model Evaluation

What are we doing in Stage 4?

We are not training anything.
 We are interrogating the reward model we just built.We want to find its blind spots.

  Its biases. The patterns it learned that are wrong or incomplete.
  
  These blind spots are not embarrassing — they are the setup for Stage 6.
  
  Every blind spot we document here becomes evidence of why reward hacking happens later.

In [ ]:
# ============================================================
# STAGE 4 — CELL 1: Full reward model evaluation
# ============================================================
import torch
import json
import numpy as np
from transformers import DistilBertTokenizer

# Load eval data — 200 examples held out since Stage 1
# These pairs were never seen during reward model training
with open('eval_data.json', 'r') as f:
    eval_data = json.load(f)

print(f"Eval examples loaded: {len(eval_data)}")

# Load tokenizer
tokenizer_rm = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# Make sure reward model is in eval mode
reward_model.eval()

def get_reward_score(text):
    """
    Takes a string (instruction + response).
    Returns a single reward score as a float.
    """
    tokens = tokenizer_rm(
        text,
        max_length=256,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = tokens['input_ids'].to("cuda")
    attention_mask = tokens['attention_mask'].to("cuda")

    with torch.no_grad():
        score = reward_model(input_ids, attention_mask)

    # .item() converts single-element tensor to Python float
    return score.item()

# Evaluate all 200 eval pairs
print("\nEvaluating all 200 pairs...")

results = []

for pair in eval_data:

    chosen_text = f"{pair['instruction']} {pair['chosen']}"
    rejected_text = f"{pair['instruction']} {pair['rejected']}"

    chosen_score = get_reward_score(chosen_text)
    rejected_score = get_reward_score(rejected_text)

    # correct = True if reward model ranked chosen above rejected
    correct = chosen_score > rejected_score

    results.append({
        'instruction': pair['instruction'],
        'chosen': pair['chosen'],
        'rejected': pair['rejected'],
        'chosen_score': chosen_score,
        'rejected_score': rejected_score,
        'score_gap_ultrafeedback': pair['score_gap'],
        'reward_diff': chosen_score - rejected_score,
        'correct': correct
    })

# Overall accuracy
overall_accuracy = sum(r['correct'] for r in results) / len(results)
print(f"\nOverall accuracy on 200 eval pairs: {overall_accuracy:.3f}")
print(f"Correct rankings: {sum(r['correct'] for r in results)}/200")

# ============================================================
# PSEUDOCODE SUMMARY:
# load 200 eval pairs from eval_data.json
# for each pair:
#     get reward score for (instruction + chosen)
#     get reward score for (instruction + rejected)
#     correct = chosen_score > rejected_score
# compute overall accuracy
# ============================================================

Eval examples loaded: 200

Evaluating all 200 pairs...

Overall accuracy on 200 eval pairs: 0.565
Correct rankings: 113/200


In [ ]:
# ============================================================
# STAGE 4 — CELL 2: Accuracy broken down by score gap
# ============================================================

# Group results by UltraFeedback score gap
# This tells us: does the reward model do better on obvious pairs?
# gap 2-3 = subtle difference
# gap 4-5 = clear difference
# gap 6+  = very obvious difference

gap_buckets = {
    'gap_2_to_3': [],
    'gap_3_to_5': [],
    'gap_5_plus': []
}

for r in results:
    gap = r['score_gap_ultrafeedback']
    if gap < 3:
        gap_buckets['gap_2_to_3'].append(r['correct'])
    elif gap < 5:
        gap_buckets['gap_3_to_5'].append(r['correct'])
    else:
        gap_buckets['gap_5_plus'].append(r['correct'])

print("Accuracy by UltraFeedback score gap:")
print("(Higher gap = more obvious quality difference)")
print()

for bucket_name, correct_list in gap_buckets.items():
    if len(correct_list) > 0:
        accuracy = sum(correct_list) / len(correct_list)
        print(f"{bucket_name}: {accuracy:.3f} accuracy ({len(correct_list)} pairs)")

# Score distributions
chosen_scores = [r['chosen_score'] for r in results]
rejected_scores = [r['rejected_score'] for r in results]

print(f"\nChosen response reward scores:")
print(f"  Mean:   {np.mean(chosen_scores):.4f}")
print(f"  Std:    {np.std(chosen_scores):.4f}")
print(f"  Min:    {np.min(chosen_scores):.4f}")
print(f"  Max:    {np.max(chosen_scores):.4f}")

print(f"\nRejected response reward scores:")
print(f"  Mean:   {np.mean(rejected_scores):.4f}")
print(f"  Std:    {np.std(rejected_scores):.4f}")
print(f"  Min:    {np.min(rejected_scores):.4f}")
print(f"  Max:    {np.max(rejected_scores):.4f}")

# How many pairs have overlapping scores
overlap = sum(1 for r in results
              if r['chosen_score'] < np.mean(rejected_scores) + np.std(rejected_scores)
              and r['rejected_score'] > np.mean(chosen_scores) - np.std(chosen_scores))

print(f"\nPairs with overlapping score regions: {overlap}/200")

# ============================================================
# PSEUDOCODE SUMMARY:
# group 200 results into 3 buckets by UltraFeedback score gap
# compute accuracy within each bucket
# compute mean, std, min, max for chosen and rejected scores
# count pairs where scores overlap between chosen and rejected
# ============================================================

Accuracy by UltraFeedback score gap:
(Higher gap = more obvious quality difference)

gap_2_to_3: 0.583 accuracy (24 pairs)
gap_3_to_5: 0.490 accuracy (98 pairs)
gap_5_plus: 0.654 accuracy (78 pairs)

Chosen response reward scores:
  Mean:   0.2752
  Std:    2.0419
  Min:    -3.7540
  Max:    4.7230

Rejected response reward scores:
  Mean:   -0.7353
  Std:    2.1812
  Min:    -4.6723
  Max:    4.4041

Pairs with overlapping score regions: 82/200


In [ ]:
# ============================================================
# STAGE 4 — CELL 3: Failure case analysis
# ============================================================

# Find all pairs where reward model got it WRONG
failures = [r for r in results if not r['correct']]
successes = [r for r in results if r['correct']]

print(f"Total failures: {len(failures)}/200")
print(f"Total successes: {len(successes)}/200")

# Sort failures by how wrong the model was
# Most wrong = chosen score much LOWER than rejected score
failures_sorted = sorted(failures, key=lambda x: x['reward_diff'])

print("\n=== TOP 5 WORST FAILURES ===")
print("(Cases where reward model was most confidently wrong)")

for i, failure in enumerate(failures_sorted[:5]):
    print(f"\n--- Failure {i+1} ---")
    print(f"Instruction: {failure['instruction'][:120]}...")
    print(f"\nCHOSEN (UltraFeedback score: {failure['score_gap_ultrafeedback']})")
    print(f"Reward model score: {failure['chosen_score']:.4f}")
    print(f"{failure['chosen'][:200]}...")
    print(f"\nREJECTED")
    print(f"Reward model score: {failure['rejected_score']:.4f}")
    print(f"{failure['rejected'][:200]}...")
    print(f"\nReward diff: {failure['reward_diff']:.4f} (negative = wrong direction)")
    print("-" * 55)

# ============================================================
# PSEUDOCODE SUMMARY:
# separate results into failures and successes
# sort failures by reward_diff (most negative = most wrong)
# print top 5 worst failures with full context
# read these carefully — find the pattern
# ============================================================

Total failures: 87/200
Total successes: 113/200

=== TOP 5 WORST FAILURES ===
(Cases where reward model was most confidently wrong)

--- Failure 1 ---
Instruction: i had a mail chat with Gabriella somtimes in december last year...

CHOSEN (UltraFeedback score: 5.0)
Reward model score: -3.0238
As an AI, I cannot access your personal information, emails, or chat history with others. If you have a specific question or need information from that conversation, I would recommend searching your e...

REJECTED
Reward model score: 3.3673
Gabriella is a name that is popular in various countries, including Italy, Portugal, and Spain, among others. However, without more specific information, such as your location, time zone, or the conte...

Reward diff: -6.3911 (negative = wrong direction)
-------------------------------------------------------

--- Failure 2 ---
Instruction: Given the following reasoning and answer, what was the question? Doris Day is a American female pop singer.
 The answer:..

In [ ]:
# ============================================================
# STAGE 4 — CELL 4: Bias analysis
# ============================================================

# BIAS 1 — Length bias
# Does the reward model prefer longer responses?
chosen_lengths = [len(r['chosen']) for r in results]
rejected_lengths = [len(r['rejected']) for r in results]

# For failures specifically — was the rejected response longer?
failure_rejected_longer = sum(
    1 for r in failures
    if len(r['rejected']) > len(r['chosen'])
)

print("=== BIAS 1: LENGTH BIAS ===")
print(f"Average chosen length:   {np.mean(chosen_lengths):.0f} chars")
print(f"Average rejected length: {np.mean(rejected_lengths):.0f} chars")
print(f"\nIn failure cases:")
print(f"Rejected was LONGER than chosen: {failure_rejected_longer}/{len(failures)}")
print(f"Percentage: {failure_rejected_longer/len(failures)*100:.1f}%")
print("(High % suggests reward model prefers longer responses)")

# BIAS 2 — Phrase bias
# Does reward model prefer responses with certain phrases?
assistant_phrases = [
    "i'm happy to help",
    "i hope this helps",
    "please note",
    "certainly",
    "of course",
    "as an ai",
    "great question"
]

print("\n=== BIAS 2: ASSISTANT PHRASE BIAS ===")
print("Checking if failures have more assistant phrases in rejected responses")

for phrase in assistant_phrases:

    # How often does this phrase appear in rejected responses
    # that the reward model incorrectly scored higher?
    phrase_in_rejected_failures = sum(
        1 for r in failures
        if phrase in r['rejected'].lower()
    )

    phrase_in_chosen_failures = sum(
        1 for r in failures
        if phrase in r['chosen'].lower()
    )

    if phrase_in_rejected_failures > 0:
        print(f"'{phrase}':")
        print(f"  In rejected (incorrectly preferred): {phrase_in_rejected_failures}")
        print(f"  In chosen (incorrectly penalized):   {phrase_in_chosen_failures}")

# BIAS 3 — Score gap vs reward model confidence
# Does the reward model do worse when UltraFeedback gap is small?
print("\n=== BIAS 3: CONFIDENCE vs DIFFICULTY ===")

reward_diffs = [r['reward_diff'] for r in results]
uf_gaps = [r['score_gap_ultrafeedback'] for r in results]

# Correlation between UltraFeedback gap and reward model diff
correlation = np.corrcoef(uf_gaps, reward_diffs)[0, 1]
print(f"Correlation between UF score gap and reward model diff: {correlation:.4f}")
print("(Positive = reward model agrees more on obvious pairs)")
print("(Near zero = reward model ignores difficulty)")

# Summary of biases found
print("\n=== BIAS SUMMARY ===")
print("These are the blind spots PPO will exploit in Stage 6:")
print()

if failure_rejected_longer/len(failures) > 0.5:
    print("⚠️  LENGTH BIAS DETECTED")
    print("   Reward model prefers longer responses")
    print("   PPO may learn to generate unnecessarily long outputs")
else:
    print("✅ No strong length bias detected")

if correlation > 0.2:
    print("✅ Reward model is sensitive to pair difficulty")
    print("   Performs better on obvious quality differences")
else:
    print("⚠️  Reward model ignores pair difficulty")
    print("   Similar performance on easy and hard pairs")

# ============================================================
# PSEUDOCODE SUMMARY:
# BIAS 1: compare lengths of chosen vs rejected in failure cases
#         high % of longer-rejected failures = length bias
# BIAS 2: check if certain assistant phrases appear more in
#         rejected responses that reward model incorrectly preferred
# BIAS 3: correlate UltraFeedback score gap with reward model diff
#         positive correlation = model is difficulty-aware
# print bias summary — these become Stage 6 predictions
# ============================================================

=== BIAS 1: LENGTH BIAS ===
Average chosen length:   926 chars
Average rejected length: 666 chars

In failure cases:
Rejected was LONGER than chosen: 42/87
Percentage: 48.3%
(High % suggests reward model prefers longer responses)

=== BIAS 2: ASSISTANT PHRASE BIAS ===
Checking if failures have more assistant phrases in rejected responses
'i'm happy to help':
  In rejected (incorrectly preferred): 1
  In chosen (incorrectly penalized):   0
'i hope this helps':
  In rejected (incorrectly preferred): 2
  In chosen (incorrectly penalized):   2
'please note':
  In rejected (incorrectly preferred): 2
  In chosen (incorrectly penalized):   2
'certainly':
  In rejected (incorrectly preferred): 1
  In chosen (incorrectly penalized):   3
'as an ai':
  In rejected (incorrectly preferred): 5
  In chosen (incorrectly penalized):   4

=== BIAS 3: CONFIDENCE vs DIFFICULTY ===
Correlation between UF score gap and reward model diff: 0.1450
(Positive = reward model agrees more on obvious pairs)
(Near ze

## Stage 4 — Deep Reward Model Evaluation — Conclusion

### What Stage 4 is telling us

**Thing 1 — The reward model is weak but not useless**
- Overall accuracy: 56.5% (113/200 pairs correct)
- Random guessing = 50%
- We are 6.5 points above random
- The model learned something but not much

**Thing 2 — The reward model only works on obvious cases**
- Easy pairs (gap 5+):    65.4% accuracy — somewhat works
- Medium pairs (gap 3-5): 49.0% accuracy — WORSE THAN RANDOM
- Hard pairs (gap 2-3):   58.3% accuracy — barely works
- 98 out of 200 eval pairs fall in the medium bucket
- On those 98 pairs the reward model is actively wrong more than right

**Thing 3 — The reward model learned the wrong things**
The 5 failure cases revealed the real problem.
The reward model was fooled by:
- Responses starting with "Sure! I'd be happy to help!" → scored high even when content is wrong
- Responses that sound confident → rewards surface confidence over correctness
- Honest refusals → penalizes "I cannot do that" even when refusal is correct
- It learned tone and style patterns instead of actual quality judgment

**Thing 4 — These blind spots are our Stage 6 predictions**
PPO in Stage 6 will exploit exactly these blind spots.
Predictions:
- PPO will learn to start responses with enthusiastic assistant phrases
- PPO will learn to avoid honest refusals
- PPO will learn to sound confident regardless of correctness
Stage 4 gave us the map of weaknesses.
Stage 6 will show PPO finding and exploiting that map.

### Score distributions
- Chosen mean reward score:   0.2752
- Rejected mean reward score: -0.7353
- Overlapping score pairs:    82/200
- Both distributions have huge std (~2.0) — wide overlap

### Bias analysis results
- Length bias:          NOT detected (48.3% — nearly 50/50)
- Main bias:            Confident surface tone
- Secondary bias:       Refusal penalty
- Difficulty awareness: Very low (correlation = 0.145)

### Project notebook update
| Experiment                  | Result                              |
|-----------------------------|-------------------------------------|
| Eval accuracy overall       | 56.5%                               |
| Accuracy gap 2-3            | 58.3%                               |
| Accuracy gap 3-5            | 49.0% (worse than random)           |
| Accuracy gap 5+             | 65.4%                               |
| Chosen mean reward score    | 0.2752                              |
| Rejected mean reward score  | -0.7353                             |
| Overlapping score pairs     | 82/200                              |
| Main bias found             | Confident surface tone              |
| Secondary bias              | Refusal penalty                     |
| Length bias                 | Not detected                        |
| Difficulty awareness        | Very low (corr = 0.145)             |
| Stage 6 prediction          | PPO will learn assistant phrases    |
|                             | and avoid honest refusals           |

### One sentence summary
"Our reward model learned surface tone patterns instead of actual quality,
and those patterns are exactly what PPO will exploit in Stage 6."

"The reward model achieved 56.5% overall accuracy — 6.5 points above random. On high-gap obvious pairs it reached 65.4%, showing it learned some meaningful signal. But on medium difficulty pairs, which made up nearly half the eval set, accuracy dropped to 49% — worse than random guessing. This happened because the model learned surface tone patterns — enthusiastic openings, stated confidence, assistant-style phrasing — rather than actual quality judgment. Reading the failure cases confirmed this: it consistently preferred responses that sounded helpful over responses that were actually correct, and penalized honest refusals. For this project that imperfection is valuable — those specific biases become testable predictions for Stage 6.

# Stage 5 — PPO Training

What are we doing in Stage 5?

We are taking the SFT model and training it using Reinforcement Learning. Specifically using PPO — Proximal Policy Optimization.

After this stage, GPT-2 will have been trained not by copying good examples (like SFT) but by trying things, getting scored by our reward model, and updating itself to get higher scores.

Why does PPO need 4 models?Model 1 — Policy model:     the GPT-2 we are training (changes every step)

Model 2 — Reference model:  the original SFT GPT-2 (frozen, never changes)


Model 3 — Reward model:     our trained DistilBERT judge (frozen)


Model 4 — Value model:
     value head on top of policy modelModels 3 and 4 are actually attached to existing models. The reward model is our DistilBERT. The value head is part of the policy model. So technically we have 2 separate model objects but 4 functional components.

In [ ]:
# ============================================================
# STAGE 5 — DIAGNOSTIC: Check TRL version and available classes
# ============================================================

import trl
print(f"TRL version: {trl.__version__}")

# Check what is actually available in this TRL version
print("\nAvailable trainers in TRL:")
import inspect
trl_contents = dir(trl)
trainers = [x for x in trl_contents if 'Trainer' in x or 'PPO' in x or 'Config' in x]
for t in trainers:
    print(f"  {t}")

TRL version: 1.6.0

Available trainers in TRL:
  DPOConfig
  DPOTrainer
  DatasetMixtureConfig
  GRPOConfig
  GRPOTrainer
  KTOConfig
  KTOTrainer
  ModelConfig
  RLOOConfig
  RLOOTrainer
  RewardConfig
  RewardTrainer
  SFTConfig
  SFTTrainer


In [ ]:
# Reinstall TRL 1.6.0
!pip install -q trl==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.6 MB/s eta 0:00:00


In [ ]:
# ============================================================
# PRE-RESTART BACKUP: Save everything before runtime restart
# ============================================================
import torch
import shutil
from google.colab import drive

drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/deccan_ai_project', exist_ok=True)

# Save reward model
torch.save(
    reward_model.state_dict(),
    '/content/drive/MyDrive/deccan_ai_project/reward_model.pt'
)
print("✅ reward_model.pt saved")

# Save SFT model
shutil.copytree(
    './sft_model',
    '/content/drive/MyDrive/deccan_ai_project/sft_model',
    dirs_exist_ok=True
)
print("✅ sft_model saved")

# Save data files
for filename in ['sft_data.json', 'preference_data.json', 'eval_data.json']:
    shutil.copy(
        filename,
        f'/content/drive/MyDrive/deccan_ai_project/{filename}'
    )
    print(f"✅ {filename} saved")

print("\n✅ Everything backed up. Safe to restart runtime now.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ reward_model.pt saved
✅ sft_model saved
✅ sft_data.json saved
✅ preference_data.json saved
✅ eval_data.json saved

✅ Everything backed up. Safe to restart runtime now.


In [ ]:
# ============================================================
# POST-RESTART RECOVERY CELL
# Run this FIRST after any runtime restart
# ============================================================
import torch
import torch.nn as nn
import shutil
import os
import json
from google.colab import drive
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    DistilBertModel,
    DistilBertTokenizer
)

# Mount Drive
drive.mount('/content/drive')

# Restore files from Drive to local storage
os.makedirs('./sft_model', exist_ok=True)
shutil.copytree(
    '/content/drive/MyDrive/deccan_ai_project/sft_model',
    './sft_model',
    dirs_exist_ok=True
)

for filename in ['sft_data.json', 'preference_data.json', 'eval_data.json']:
    shutil.copy(
        f'/content/drive/MyDrive/deccan_ai_project/{filename}',
        filename
    )

print("✅ Files restored from Drive")

# Rebuild RewardModel class
class RewardModel(nn.Module):
    def __init__(self):
        super(RewardModel, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.reward_head = nn.Linear(768, 1)
        self.dropout = nn.Dropout(0.1)

    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        reward = self.reward_head(cls_output)
        return reward.squeeze(-1)

# Load reward model weights
reward_model = RewardModel()
reward_model.load_state_dict(
    torch.load('/content/drive/MyDrive/deccan_ai_project/reward_model.pt')
)
reward_model = reward_model.to("cuda")
reward_model.eval()
print("✅ Reward model loaded")

# Load tokenizers
tokenizer_rm = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
tokenizer_ppo = GPT2Tokenizer.from_pretrained("./sft_model")
tokenizer_ppo.pad_token = tokenizer_ppo.eos_token
print("✅ Tokenizers loaded")

# Verify TRL version
import trl
print(f"\nTRL version: {trl.__version__}")
print(f"Available: {[x for x in dir(trl) if 'Trainer' in x]}")

# Memory check
print(f"\nVRAM used: {torch.cuda.memory_allocated(0)/(1024**3):.3f} GB")
print(f"VRAM free: {(14.56 - torch.cuda.memory_allocated(0)/(1024**3)):.3f} GB")
print("\n✅ Recovery complete. Ready to continue.")

Mounted at /content/drive
✅ Files restored from Drive


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

✅ Reward model loaded


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

✅ Tokenizers loaded

TRL version: 1.6.0
Available: ['DPOTrainer', 'GRPOTrainer', 'KTOTrainer', 'RLOOTrainer', 'RewardTrainer', 'SFTTrainer']

VRAM used: 0.248 GB
VRAM free: 14.312 GB

✅ Recovery complete. Ready to continue.


In [ ]:
# ============================================================
# STAGE 5 — CELL 1: Load policy model for GRPO
# ============================================================
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from trl import GRPOConfig, GRPOTrainer

torch.cuda.empty_cache()

print("Loading SFT model as policy...")
policy_model = GPT2LMHeadModel.from_pretrained("./sft_model")
policy_model = policy_model.to("cuda")

memory_used = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM after loading policy: {memory_used:.3f} GB")
print(f"VRAM free: {(14.56 - memory_used):.3f} GB")

# Count parameters
total = sum(p.numel() for p in policy_model.parameters())
print(f"Policy model parameters: {total:,}")

print("\n✅ Policy model loaded")
print("Note: GRPO does not need a value head")
print("Note: GRPO does not need a separate value model")
print("This saves ~0.5GB vs PPO setup")

# ============================================================
# PSEUDOCODE SUMMARY:
# load SFT model as policy model
# no value head needed — GRPO eliminates value model entirely
# check VRAM — should have 13+ GB free
# ============================================================

Loading SFT model as policy...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

VRAM after loading policy: 0.713 GB
VRAM free: 13.847 GB
Policy model parameters: 124,439,808

✅ Policy model loaded
Note: GRPO does not need a value head
Note: GRPO does not need a separate value model
This saves ~0.5GB vs PPO setup


In [ ]:
# ============================================================
# STAGE 5 — CELL 2: Configure GRPO
# ============================================================
import json
import torch
from transformers import DistilBertTokenizer
from trl import GRPOConfig, GRPOTrainer

# Load 500 prompts for GRPO training
with open('sft_data.json', 'r') as f:
    sft_data_full = json.load(f)

# We only need the instruction — GRPO generates its own responses
ppo_prompts = [ex['instruction'] for ex in sft_data_full[:500]]

print(f"Training prompts loaded: {len(ppo_prompts)}")
print(f"Sample: {ppo_prompts[0][:80]}...")

# Convert to HuggingFace dataset format
# GRPOTrainer expects a dataset with a "prompt" column
from datasets import Dataset
prompt_dataset = Dataset.from_dict({"prompt": ppo_prompts})
print(f"Dataset created: {len(prompt_dataset)} examples")

# ============================================================
# REWARD FUNCTION
# This is the key difference from PPO
# In PPO: reward model is called by the trainer internally
# In GRPO: we define a Python function that takes
#          a list of prompts and responses and returns rewards
# GRPO calls this function after generating responses
# ============================================================

def reward_function(prompts, completions, **kwargs):
    """
    GRPO reward function.

    prompts:     list of instruction strings
    completions: list of generated response strings

    Returns: list of reward scores (one per completion)

    This function is called by GRPOTrainer after it generates
    multiple responses per prompt. We score each response
    using our trained DistilBERT reward model.
    """
    rewards = []

    for prompt, completion in zip(prompts, completions):
        # Combine instruction and response
        # Same format as reward model training
        text = f"{prompt} {completion}"

        # Tokenize
        tokens = tokenizer_rm(
            text,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to("cuda")

        # Get reward score
        with torch.no_grad():
            score = reward_model(
                tokens['input_ids'],
                tokens['attention_mask']
            )

        # Convert tensor to float
        rewards.append(score.item())

    return rewards

# Test reward function works
test_rewards = reward_function(
    ["What is the capital of France?"],
    ["Paris is the capital of France."]
)
print(f"\nReward function test: {test_rewards}")
print("✅ Reward function works")

# ============================================================
# PSEUDOCODE SUMMARY:
# load 500 prompts from sft_data.json
# convert to HuggingFace Dataset with "prompt" column
# define reward_function:
#     takes list of prompts and completions
#     for each pair: tokenize and score with reward_model
#     return list of float reward scores
# test reward function on one example
# ============================================================

Training prompts loaded: 500
Sample: From: Reed Hastings
Sent: Sunday, August 14, 2016 6:36 PM
To: Peter Thiel
Subjec...
Dataset created: 500 examples

Reward function test: [-2.827207326889038]
✅ Reward function works


In [ ]:
# ============================================================
# STAGE 5 — CELL 3: GRPO Config fixed
# ============================================================
import torch
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    num_generations=4,
    generation_batch_size=4,
    # generation_batch_size must be divisible by num_generations
    # 4 ÷ 4 = 1 ✅
    # This means: generate 4 responses, score all 4, update once
    max_completion_length=128,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    beta=0.1,
    seed=42,
    temperature=0.7,
)

print("GRPO Configuration:")
print(f"  num_generations:       {grpo_config.num_generations}")
print(f"  generation_batch_size: {grpo_config.generation_batch_size}")
print(f"  beta (kl_coef):        {grpo_config.beta}")
print(f"  learning_rate:         {grpo_config.learning_rate}")
print(f"  max_completion_length: {grpo_config.max_completion_length}")
print(f"  epochs:                {grpo_config.num_train_epochs}")

# Create GRPO Trainer
grpo_trainer = GRPOTrainer(
    model=policy_model,
    args=grpo_config,
    reward_funcs=reward_function,
    train_dataset=prompt_dataset,
    processing_class=tokenizer_ppo,
)

print("\n✅ GRPO Trainer created")
print(f"Training examples:    {len(prompt_dataset)}")
print(f"Responses per prompt: {grpo_config.num_generations}")
print(f"Total generations:    {len(prompt_dataset) * grpo_config.num_generations}")

memory = torch.cuda.memory_allocated(0) / (1024**3)
print(f"\nVRAM before training: {memory:.3f} GB")
print(f"VRAM free: {(14.56 - memory):.3f} GB")

# ============================================================
# PSEUDOCODE SUMMARY:
# fix: generation_batch_size=4 must equal num_generations=4
# 4 responses generated per prompt
# all 4 scored together
# advantages normalized within group of 4
# one policy update per group
# ============================================================

GRPO Configuration:
  num_generations:       4
  generation_batch_size: 4
  beta (kl_coef):        0.1
  learning_rate:         1e-05
  max_completion_length: 128
  epochs:                1


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


✅ GRPO Trainer created
Training examples:    500
Responses per prompt: 4
Total generations:    2000

VRAM before training: 1.186 GB
VRAM free: 13.374 GB


GRPO Trainer created

successfully — no errors this time.VRAM: 1.186 GB used, 13.374 GB free — very comfortable. GRPO only needs 2 model copies (policy + reference).

 Much lighter than PPO's 4 components.Total generations: 2000 — 500 prompts × 4 responses each = 2000 total response generations in one epoch.

 Reward function test: -2.827 — the Reed Hastings email prompt got a negative reward score. Makes sense — our reward model was trained on simple QA pairs, not complex email tasks.

 Negative reward just means "below average." The model will learn to do better than this baseline.

In [ ]:
# ============================================================
# POST-RESTART RECOVERY — STAGE 5
# ============================================================
import torch
import torch.nn as nn
import shutil
import os
import json
from google.colab import drive
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    DistilBertModel,
    DistilBertTokenizer
)
from datasets import Dataset
from trl import GRPOConfig, GRPOTrainer

# Mount Drive
drive.mount('/content/drive')

# Restore files
os.makedirs('./sft_model', exist_ok=True)
shutil.copytree(
    '/content/drive/MyDrive/deccan_ai_project/sft_model',
    './sft_model',
    dirs_exist_ok=True
)
for filename in ['sft_data.json', 'preference_data.json', 'eval_data.json']:
    shutil.copy(
        f'/content/drive/MyDrive/deccan_ai_project/{filename}',
        filename
    )
print(" Files restored")

# Rebuild RewardModel
class RewardModel(nn.Module):
    def __init__(self):
        super(RewardModel, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.reward_head = nn.Linear(768, 1)
        self.dropout = nn.Dropout(0.1)
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        cls_output = outputs.last_hidden_state[:, 0, :]
        cls_output = self.dropout(cls_output)
        reward = self.reward_head(cls_output)
        return reward.squeeze(-1)

reward_model = RewardModel()
reward_model.load_state_dict(
    torch.load('/content/drive/MyDrive/deccan_ai_project/reward_model.pt')
)
reward_model = reward_model.to("cuda")
reward_model.eval()
print(" Reward model loaded")

# Load tokenizers
tokenizer_rm = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
tokenizer_ppo = GPT2Tokenizer.from_pretrained("./sft_model")
tokenizer_ppo.pad_token = tokenizer_ppo.eos_token
print(" Tokenizers loaded")

# Load policy model
policy_model = GPT2LMHeadModel.from_pretrained("./sft_model")
policy_model = policy_model.to("cuda")
print(" Policy model loaded")

# Filter short prompts
with open('sft_data.json', 'r') as f:
    sft_data_full = json.load(f)

short_prompts = []
for ex in sft_data_full:
    tokens = tokenizer_ppo(ex['instruction'], truncation=False)
    if len(tokens['input_ids']) <= 256:
        short_prompts.append(ex['instruction'])
    if len(short_prompts) >= 300:
        break

prompt_dataset = Dataset.from_dict({"prompt": short_prompts})
print(f" Filtered prompts: {len(prompt_dataset)}")

# Reward function
def reward_function(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = f"{prompt} {completion}"
        tokens = tokenizer_rm(
            text,
            max_length=256,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        ).to("cuda")
        with torch.no_grad():
            score = reward_model(
                tokens['input_ids'],
                tokens['attention_mask']
            )
        rewards.append(score.item())
    return rewards

# Test reward function
test = reward_function(
    ["What is gravity?"],
    ["Gravity is a force that pulls objects together."]
)
print(f" Reward function test: {test}")

# GRPO Config
grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    num_generations=4,
    generation_batch_size=4,
    max_completion_length=100,
    max_prompt_length=256,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    beta=0.1,
    seed=42,
    temperature=0.7,
)

# GRPO Trainer
grpo_trainer = GRPOTrainer(
    model=policy_model,
    args=grpo_config,
    reward_funcs=reward_function,
    train_dataset=prompt_dataset,
    processing_class=tokenizer_ppo,
)

print(f"\n GRPO Trainer created")
print(f"Training prompts: {len(prompt_dataset)}")
print(f"Total generations: {len(prompt_dataset) * grpo_config.num_generations}")

memory = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM: {memory:.3f} GB used, {(14.56-memory):.3f} GB free")
print("\nFull recovery complete. Ready to train.")

Mounted at /content/drive
 Files restored


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

...

 Tokenizers loaded


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1066 > 1024). Running this sequence through the model will result in indexing errors


 Policy model loaded
 Filtered prompts: 300
 Reward function test: [-1.5871797800064087]


TypeError: GRPOConfig.__init__() got an unexpected keyword argument 'max_prompt_length'

In [ ]:
!pip install -q trl==1.6.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.1/825.1 kB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.4 MB/s eta 0:00:00


In [ ]:
# ============================================================
# STAGE 5 — GRPO Config without max_prompt_length
# ============================================================
grpo_config = GRPOConfig(
    output_dir="./grpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    num_generations=4,
    generation_batch_size=4,
    max_completion_length=80,
    # reduced further for safety
    # 256 prompt tokens + 80 completion = 336 total
    # well within 1024 GPT-2 limit
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    beta=0.1,
    seed=42,
    temperature=0.7,
)

print("GRPO Config created")
print(f"  max_completion_length: {grpo_config.max_completion_length}")
print(f"  num_generations:       {grpo_config.num_generations}")
print(f"  beta:                  {grpo_config.beta}")

# GRPO Trainer
grpo_trainer = GRPOTrainer(
    model=policy_model,
    args=grpo_config,
    reward_funcs=reward_function,
    train_dataset=prompt_dataset,
    processing_class=tokenizer_ppo,
)

print(f"\n✅ GRPO Trainer created")
print(f"Training prompts:  {len(prompt_dataset)}")
print(f"Total generations: {len(prompt_dataset) * grpo_config.num_generations}")

memory = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM: {memory:.3f} GB used, {(14.56-memory):.3f} GB free")

GRPO Config created
  max_completion_length: 80
  num_generations:       4
  beta:                  0.1


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


✅ GRPO Trainer created
Training prompts:  300
Total generations: 1200
VRAM: 1.186 GB used, 13.374 GB free


In [ ]:
# ============================================================
# STAGE 5 — CELL 4: Run GRPO training
# ============================================================
import time

print("Starting GRPO training...")
print("=" * 55)
print("Watch every 10 steps:")
print("  reward/mean — average reward across 4 responses")
print("  kl          — drift from SFT reference model")
print("  loss        — policy gradient loss")
print("=" * 55)

start_time = time.time()

grpo_trainer.train()

end_time = time.time()
training_time = (end_time - start_time) / 60

print(f"\n✅ GRPO Training Complete!")
print(f"Training time: {training_time:.1f} minutes")

# Save trained model
grpo_trainer.save_model("./grpo_model")
tokenizer_ppo.save_pretrained("./grpo_model")
print("Model saved to ./grpo_model")

memory_after = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM after training: {memory_after:.3f} GB")

# ============================================================
# PSEUDOCODE SUMMARY:
# grpo_trainer.train() runs:
#     for each prompt (300 total):
#         generate 4 responses
#         score all 4 with reward_function
#         group_mean = average of 4 scores
#         advantage = (score - group_mean) / std
#         responses above mean → reinforced
#         responses below mean → discouraged
#         KL penalty (beta=0.1) prevents drift from SFT
#         update policy weights
# save final model
# ============================================================

Starting GRPO training...
Watch every 10 steps:
  reward/mean — average reward across 4 responses
  kl          — drift from SFT reference model
  loss        — policy gradient loss


Step,Training Loss
10,0.151602
20,1.468462
30,0.849638
40,24.086212
50,0.219103
60,0.172179
70,0.488125
80,0.263426
90,0.148335
100,1.312379



✅ GRPO Training Complete!
Training time: 11.5 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./grpo_model
VRAM after training: 2.125 GB


In [ ]:
# ============================================================
# STAGE 5 — CELL 5: Test GRPO model outputs
# ============================================================
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load GRPO trained model
grpo_model = GPT2LMHeadModel.from_pretrained("./grpo_model")
grpo_model = grpo_model.to("cuda")
grpo_model.eval()

grpo_tokenizer = GPT2Tokenizer.from_pretrained("./grpo_model")
grpo_tokenizer.pad_token = grpo_tokenizer.eos_token

print("GRPO model loaded")

def generate_grpo(model, tokenizer, instruction, max_new_tokens=100):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    full_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:" in full_output:
        response = full_output.split("### Response:")[-1].strip()
    else:
        response = full_output
    return response

# Same 5 test prompts as SFT evaluation
test_prompts = [
    "What is the capital of France?",
    "Explain what machine learning is in simple terms.",
    "Write a short poem about the ocean.",
    "What are the benefits of exercise?",
    "How do you make a cup of tea?"
]

print("\n" + "=" * 55)
print("GRPO MODEL OUTPUT QUALITY TEST")
print("=" * 55)

for i, prompt in enumerate(test_prompts):
    print(f"\n--- Test {i+1} ---")
    print(f"Instruction: {prompt}")
    response = generate_grpo(grpo_model, grpo_tokenizer, prompt)
    print(f"Response: {response}")
    print("-" * 55)

# Also score outputs with reward model
print("\n" + "=" * 55)
print("REWARD SCORES — GRPO vs SFT")
print("=" * 55)

# Load SFT model for comparison
sft_model_compare = GPT2LMHeadModel.from_pretrained("./sft_model")
sft_model_compare = sft_model_compare.to("cuda")
sft_model_compare.eval()
sft_tok = GPT2Tokenizer.from_pretrained("./sft_model")
sft_tok.pad_token = sft_tok.eos_token

for prompt in test_prompts[:3]:
    grpo_response = generate_grpo(grpo_model, grpo_tokenizer, prompt)
    sft_response = generate_grpo(sft_model_compare, sft_tok, prompt)

    grpo_score = reward_function([prompt], [grpo_response])[0]
    sft_score = reward_function([prompt], [sft_response])[0]

    print(f"\nPrompt: {prompt[:60]}...")
    print(f"SFT reward score:  {sft_score:.4f}")
    print(f"GRPO reward score: {grpo_score:.4f}")
    print(f"Improvement: {grpo_score - sft_score:+.4f}")

# Clean up
del sft_model_compare
torch.cuda.empty_cache()

# ============================================================
# PSEUDOCODE SUMMARY:
# load grpo_model from ./grpo_model
# generate responses to same 5 test prompts as SFT
# compare output quality visually
# score first 3 prompts with reward model
# compare GRPO reward scores vs SFT reward scores
# if GRPO scores higher → RL training worked
# ============================================================

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GRPO model loaded

GRPO MODEL OUTPUT QUALITY TEST

--- Test 1 ---
Instruction: What is the capital of France?
Response: France has a population that roughly corresponds to about 20 million people. This means there are more than 4,000 different languages spoken by French speakers every day in this country and most inhabitants speak English or Spanish with accents similar either from their home countries (eastern Europe) or abroad (Europe). The total number living outside of these borders represents approximately 1 percent comparedto 5% across all other regions but does not represent 100 % native income level for some parts because it excludes immigrants as well as natives born overseas
-------------------------------------------------------

--- Test 2 ---
Instruction: Explain what machine learning is in simple terms.
Response: Machine Learning can be used to identify trends, predict future events, or analyze complex data structures from a variety of sources such as social media and bus

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Prompt: What is the capital of France?...
SFT reward score:  -0.6316
GRPO reward score: -2.9546
Improvement: -2.3231

Prompt: Explain what machine learning is in simple terms....
SFT reward score:  -1.2016
GRPO reward score: -2.5475
Improvement: -1.3459

Prompt: Write a short poem about the ocean....
SFT reward score:  -1.8299
GRPO reward score: -3.2492
Improvement: -1.4193


In [ ]:
# ============================================================
# STAGE 5 — CELL 6: Reward hacking and bias analysis
# ============================================================
import torch
import json
import numpy as np

# Load both models
print("Loading models for comparison...")
from transformers import GPT2LMHeadModel, GPT2Tokenizer

sft_model = GPT2LMHeadModel.from_pretrained("./sft_model")
sft_model = sft_model.to("cuda")
sft_model.eval()

grpo_model = GPT2LMHeadModel.from_pretrained("./grpo_model")
grpo_model = grpo_model.to("cuda")
grpo_model.eval()

tokenizer_gen = GPT2Tokenizer.from_pretrained("./sft_model")
tokenizer_gen.pad_token = tokenizer_gen.eos_token

def generate(model, tokenizer, instruction, max_new_tokens=80):
    prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=200
    ).to("cuda")
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "### Response:" in full:
        return full.split("### Response:")[-1].strip()
    return full

# 20 test prompts for broad comparison
with open('sft_data.json', 'r') as f:
    sft_data = json.load(f)

# Use first 20 short prompts from sft_data
test_prompts_20 = []
for ex in sft_data:
    tokens = tokenizer_gen(ex['instruction'], truncation=False)
    if len(tokens['input_ids']) <= 100:
        test_prompts_20.append(ex['instruction'])
    if len(test_prompts_20) >= 20:
        break

print(f"Test prompts: {len(test_prompts_20)}")

# ============================================================
# EXPERIMENT A — Reward comparison on 20 prompts
# ============================================================
print("\n" + "=" * 55)
print("EXPERIMENT A: Reward scores SFT vs GRPO (20 prompts)")
print("=" * 55)

sft_rewards = []
grpo_rewards = []

for prompt in test_prompts_20:
    sft_resp = generate(sft_model, tokenizer_gen, prompt)
    grpo_resp = generate(grpo_model, tokenizer_gen, prompt)

    sft_score = reward_function([prompt], [sft_resp])[0]
    grpo_score = reward_function([prompt], [grpo_resp])[0]

    sft_rewards.append(sft_score)
    grpo_rewards.append(grpo_score)

print(f"SFT  mean reward:  {np.mean(sft_rewards):.4f}")
print(f"GRPO mean reward:  {np.mean(grpo_rewards):.4f}")
print(f"Difference:        {np.mean(grpo_rewards) - np.mean(sft_rewards):+.4f}")
print(f"GRPO better on:    {sum(g > s for g,s in zip(grpo_rewards, sft_rewards))}/20 prompts")
print(f"SFT  better on:    {sum(s > g for g,s in zip(grpo_rewards, sft_rewards))}/20 prompts")

# ============================================================
# EXPERIMENT B — Assistant phrase bias analysis
# ============================================================
print("\n" + "=" * 55)
print("EXPERIMENT B: Assistant phrase counts SFT vs GRPO")
print("=" * 55)

bias_phrases = [
    "sure",
    "i'd be happy to help",
    "certainly",
    "of course",
    "i'm happy to help",
    "great question",
    "as an ai",
    "confidence"
]

sft_responses_all = []
grpo_responses_all = []

for prompt in test_prompts_20:
    sft_responses_all.append(generate(sft_model, tokenizer_gen, prompt).lower())
    grpo_responses_all.append(generate(grpo_model, tokenizer_gen, prompt).lower())

print(f"{'Phrase':<25} {'SFT count':>10} {'GRPO count':>10} {'Change':>10}")
print("-" * 55)

for phrase in bias_phrases:
    sft_count = sum(1 for r in sft_responses_all if phrase in r)
    grpo_count = sum(1 for r in grpo_responses_all if phrase in r)
    change = grpo_count - sft_count
    direction = "↑" if change > 0 else ("↓" if change < 0 else "=")
    print(f"{phrase:<25} {sft_count:>10} {grpo_count:>10} {direction}{abs(change):>8}")

# ============================================================
# EXPERIMENT C — 5 side by side examples
# ============================================================
print("\n" + "=" * 55)
print("EXPERIMENT C: Side by side examples")
print("=" * 55)

for i in range(5):
    prompt = test_prompts_20[i]
    sft_r = generate(sft_model, tokenizer_gen, prompt)
    grpo_r = generate(grpo_model, tokenizer_gen, prompt)
    sft_score = reward_function([prompt], [sft_r])[0]
    grpo_score = reward_function([prompt], [grpo_r])[0]

    print(f"\n--- Example {i+1} ---")
    print(f"Prompt: {prompt[:80]}...")
    print(f"\nSFT  (reward {sft_score:.3f}): {sft_r[:150]}")
    print(f"\nGRPO (reward {grpo_score:.3f}): {grpo_r[:150]}")
    print("-" * 55)

# ============================================================
# PSEUDOCODE SUMMARY:
# experiment A: generate 20 responses each from SFT and GRPO
#               score all with reward model
#               compare mean rewards and per-prompt wins
# experiment B: count assistant phrases in all responses
#               SFT vs GRPO counts — did GRPO learn phrase bias?
# experiment C: print 5 side by side examples with reward scores
# ============================================================

Loading models for comparison...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1066 > 1024). Running this sequence through the model will result in indexing errors


Test prompts: 20

EXPERIMENT A: Reward scores SFT vs GRPO (20 prompts)
SFT  mean reward:  -2.1441
GRPO mean reward:  -1.9574
Difference:        +0.1867
GRPO better on:    15/20 prompts
SFT  better on:    5/20 prompts

EXPERIMENT B: Assistant phrase counts SFT vs GRPO
Phrase                     SFT count GRPO count     Change
-------------------------------------------------------
sure                               1          5 ↑       4
i'd be happy to help               0          0 =       0
certainly                          0          0 =       0
of course                          0          1 ↑       1
i'm happy to help                  0          0 =       0
great question                     0          0 =       0
as an ai                           0          0 =       0
confidence                         0          0 =       0

EXPERIMENT C: Side by side examples

--- Example 1 ---
Prompt: I need your help to write an article. The topic is about deed of novation. If yo...

SFT 

## Lessons Learned — Stage 5

Lesson 1: Reward model accuracy is not enough
  56.5% accuracy sounds modest but the real problem
  is systematic bias, not random errors.

Lesson 2: Systematic biases matter more than aggregate metrics
  The phrase bias we found in Stage 4 directly caused
  measurable behavior change in Stage 5.

Lesson 3: Policy optimization amplifies reward model weaknesses
  GRPO didn't just fail to improve — it actively learned
  to exploit the reward model's blind spots.

Lesson 4: RLHF performance is fundamentally limited by reward quality
  No amount of RL training fixes a broken reward signal.
  Garbage in, garbage out — but amplified.

Lesson 5: Verifiable rewards eliminate this problem entirely
  Code passes or fails. Math is correct or incorrect.
  These cannot be gamed by surface patterns.
  This is why STARK uses verifiable rewards.

In [ ]:
# ============================================================
# STAGE 5 — BACKUP: Save GRPO model to Drive
# ============================================================
import shutil
from google.colab import drive

drive.mount('/content/drive')

# Save GRPO model
shutil.copytree(
    './grpo_model',
    '/content/drive/MyDrive/deccan_ai_project/grpo_model',
    dirs_exist_ok=True
)
print("✅ grpo_model saved to Drive")

# Save reward model again (in case)
import torch
torch.save(
    reward_model.state_dict(),
    '/content/drive/MyDrive/deccan_ai_project/reward_model.pt'
)
print("✅ reward_model.pt saved to Drive")

print("\n✅ Stage 5 fully backed up")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ grpo_model saved to Drive
✅ reward_model.pt saved to Drive

✅ Stage 5 fully backed up


# Stage 6 — KL Penalty Experiment (Tiny Version)

In [ ]:
# ============================================================
# STAGE 6 — KL PENALTY EXPERIMENT
# beta=0.0 vs beta=0.1 on 50 prompts, 50 steps
# ============================================================
import torch
import numpy as np
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from trl import GRPOConfig, GRPOTrainer
from datasets import Dataset
import json

# We start from the SFT model fresh for both experiments
# This ensures fair comparison — same starting point
# beta=0.1 results we already have from Stage 5
# Now we run beta=0.0 from same starting point

print("Loading fresh SFT model for beta=0.0 experiment...")
model_no_kl = GPT2LMHeadModel.from_pretrained("./sft_model")
model_no_kl = model_no_kl.to("cuda")
print("✅ Fresh SFT model loaded")

# Use same 50 short prompts
with open('sft_data.json', 'r') as f:
    sft_data = json.load(f)

short_prompts_50 = []
for ex in sft_data:
    tokens = tokenizer_gen(ex['instruction'], truncation=False)
    if len(tokens['input_ids']) <= 100:
        short_prompts_50.append(ex['instruction'])
    if len(short_prompts_50) >= 50:
        break

prompt_dataset_50 = Dataset.from_dict({"prompt": short_prompts_50})
print(f"Prompts for experiment: {len(prompt_dataset_50)}")

# GRPO config with beta=0.0 — NO KL PENALTY
grpo_config_no_kl = GRPOConfig(
    output_dir="./grpo_no_kl",
    learning_rate=1e-5,
    per_device_train_batch_size=1,
    num_generations=4,
    generation_batch_size=4,
    max_completion_length=80,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
    beta=0.0,
    # THIS IS THE KEY CHANGE
    # beta=0.0 means NO KL penalty
    # total reward = reward_score - 0.0 × KL = reward_score only
    # policy can drift as far as it wants from SFT reference
    # no leash
    seed=42,
    temperature=0.7,
    max_steps=50,
    # only 50 steps — tiny experiment
    # enough to see divergence if it happens
)

print(f"\nRunning with beta=0.0 (NO KL PENALTY)")
print(f"Steps: 50")
print(f"Prompts: 50")

grpo_trainer_no_kl = GRPOTrainer(
    model=model_no_kl,
    args=grpo_config_no_kl,
    reward_funcs=reward_function,
    train_dataset=prompt_dataset_50,
    processing_class=tokenizer_ppo,
)

import time
start = time.time()
grpo_trainer_no_kl.train()
elapsed = (time.time() - start) / 60
print(f"\n✅ beta=0.0 training complete: {elapsed:.1f} minutes")

# ============================================================
# COMPARE beta=0.0 vs beta=0.1
# ============================================================
print("\n" + "=" * 55)
print("COMPARISON: beta=0.0 vs beta=0.1")
print("=" * 55)

# Generate responses from no-KL model
model_no_kl.eval()

bias_phrases = ["sure", "of course", "certainly",
                "i'm happy to help", "i'd be happy"]

# Test on 10 prompts
test_10 = short_prompts_50[:10]

no_kl_rewards = []
no_kl_responses = []

with_kl_rewards = []
with_kl_responses = []

for prompt in test_10:
    # No KL response
    no_kl_resp = generate(model_no_kl, tokenizer_gen, prompt)
    no_kl_score = reward_function([prompt], [no_kl_resp])[0]
    no_kl_rewards.append(no_kl_score)
    no_kl_responses.append(no_kl_resp.lower())

    # With KL response (our Stage 5 GRPO model)
    with_kl_resp = generate(grpo_model, tokenizer_gen, prompt)
    with_kl_score = reward_function([prompt], [with_kl_resp])[0]
    with_kl_rewards.append(with_kl_score)
    with_kl_responses.append(with_kl_resp.lower())

print(f"Mean reward beta=0.0: {np.mean(no_kl_rewards):.4f}")
print(f"Mean reward beta=0.1: {np.mean(with_kl_rewards):.4f}")
print(f"Reward difference:    {np.mean(no_kl_rewards) - np.mean(with_kl_rewards):+.4f}")

# Phrase counts
print(f"\n{'Phrase':<25} {'beta=0.0':>10} {'beta=0.1':>10}")
print("-" * 45)
for phrase in bias_phrases:
    no_kl_count = sum(1 for r in no_kl_responses if phrase in r)
    with_kl_count = sum(1 for r in with_kl_responses if phrase in r)
    print(f"{phrase:<25} {no_kl_count:>10} {with_kl_count:>10}")

# 3 side by side examples
print(f"\n3 SIDE BY SIDE EXAMPLES")
print("=" * 55)
for i in range(3):
    prompt = test_10[i]
    no_kl_r = generate(model_no_kl, tokenizer_gen, prompt)
    with_kl_r = generate(grpo_model, tokenizer_gen, prompt)
    no_kl_s = reward_function([prompt], [no_kl_r])[0]
    with_kl_s = reward_function([prompt], [with_kl_r])[0]

    print(f"\nPrompt: {prompt[:80]}...")
    print(f"beta=0.0 (reward {no_kl_s:.3f}): {no_kl_r[:150]}")
    print(f"beta=0.1 (reward {with_kl_s:.3f}): {with_kl_r[:150]}")
    print("-" * 55)

# ============================================================
# PSEUDOCODE SUMMARY:
# load fresh SFT model → train with beta=0.0 for 50 steps
# load GRPO model from Stage 5 (trained with beta=0.1)
# compare on 10 prompts:
#     mean reward scores
#     assistant phrase counts
#     3 side by side examples
# document: does removing KL increase reward? increase phrase bias?
# ============================================================

Loading fresh SFT model for beta=0.0 experiment...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

✅ Fresh SFT model loaded
Prompts for experiment: 50

Running with beta=0.0 (NO KL PENALTY)
Steps: 50
Prompts: 50


Step,Training Loss
10,0.118027
20,0.003579
30,50.580536
40,-0.007411
50,0.128003



✅ beta=0.0 training complete: 0.5 minutes

COMPARISON: beta=0.0 vs beta=0.1
Mean reward beta=0.0: -2.5754
Mean reward beta=0.1: -2.3051
Reward difference:    -0.2703

Phrase                      beta=0.0   beta=0.1
---------------------------------------------
sure                               1          1
of course                          0          0
certainly                          0          0
i'm happy to help                  0          0
i'd be happy                       0          0

3 SIDE BY SIDE EXAMPLES

Prompt: I need your help to write an article. The topic is about deed of novation. If yo...
beta=0.0 (reward -4.169): Sure! Here's a list of some questions that I think are pertinent and useful for writing articles on this subject in general. Feel free not only ask bu
beta=0.1 (reward -3.119): Yes! Your essay has been written for people who are on a job search or work experience that involves death (e-mail me with questions) and it's not rel
--------------------------

## Stage 6 — KL Penalty Experiment — Conclusion

### Setup
- beta=0.1 (with KL): Stage 5 GRPO model
- beta=0.0 (no KL):   Fresh SFT model, 50 steps

### Results
| Metric              | beta=0.1 | beta=0.0 |
|---------------------|----------|----------|
| Mean reward         | -2.305   | -2.575   |
| Loss spike          | Yes      | Yes (50.58 at step 30) |
| Training stability  | Moderate | Unstable |
| Output quality      | Degraded | Degraded |

### Finding
Removing KL did not increase reward.
It made reward worse and training more unstable.

### Why
Two reasons working together:
1. GPT-2 Small is fragile — few bad updates destroy structure
2. Reward model is noisy — without KL, policy chases noise

### Conclusion
KL serves two purposes:
1. Prevents policy drift from SFT reference (well known)
2. Stabilizes optimization against noisy reward signal (our finding)

"Removing KL did not produce textbook reward hacking.
Instead it destabilized training. This showed that KL
serves not just as a drift constraint but as an
optimization stabilizer when reward signal is noisy."

In [ ]:
# Save Stage 6
import shutil
shutil.copytree(
    './grpo_no_kl',
    '/content/drive/MyDrive/deccan_ai_project/grpo_no_kl',
    dirs_exist_ok=True
)

print("✅ Stage 6 saved to Drive")

✅ Stage 6 saved to Drive


# Stage 7 — DPO (Direct Preference Optimization)

In [ ]:
# ============================================================
# STAGE 7 — CELL 1: Format preference data for DPO
# ============================================================
import json
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load preference pairs
with open('preference_data.json', 'r') as f:
    preference_data = json.load(f)

print(f"Preference pairs loaded: {len(preference_data)}")

# DPO requires a specific format
# Each example needs: prompt, chosen, rejected
# We format instruction as prompt
# chosen and rejected responses as is

def format_dpo_example(example):
    """
    Formats one preference pair for DPO training.
    DPOTrainer expects prompt, chosen, rejected as separate fields.
    """
    return {
        'prompt': f"### Instruction:\n{example['instruction']}\n\n### Response:\n",
        'chosen': example['chosen'],
        'rejected': example['rejected']
    }

dpo_data = [format_dpo_example(ex) for ex in preference_data]

print(f"Formatted DPO examples: {len(dpo_data)}")
print(f"\nFirst example structure:")
print(f"Prompt: {dpo_data[0]['prompt'][:100]}...")
print(f"Chosen (first 100 chars): {dpo_data[0]['chosen'][:100]}...")
print(f"Rejected (first 100 chars): {dpo_data[0]['rejected'][:100]}...")

# Convert to HuggingFace Dataset
from datasets import Dataset

# Split 900 train / 100 val
import random
random.seed(42)
random.shuffle(dpo_data)

dpo_train = Dataset.from_list(dpo_data[:900])
dpo_val = Dataset.from_list(dpo_data[900:])

print(f"\nDPO train: {len(dpo_train)}")
print(f"DPO val:   {len(dpo_val)}")

# ============================================================
# PSEUDOCODE SUMMARY:
# load 1000 preference pairs from preference_data.json
# format each as: prompt, chosen, rejected
# prompt = "### Instruction:\n{instruction}\n\n### Response:\n"
# shuffle with seed 42
# split 900 train / 100 validation
# convert to HuggingFace Dataset
# ============================================================

Preference pairs loaded: 1000
Formatted DPO examples: 1000

First example structure:
Prompt: ### Instruction:
Can you use your reasoning skills to complete this puzzle? Fill in the blanks with ...
Chosen (first 100 chars): Yes, I can help you complete the puzzle. The word you are looking for is "AID." The news article tit...
Rejected (first 100 chars): Hello! I'm here to help you with your question. I understand that you want me to complete a puzzle r...

DPO train: 900
DPO val:   100


In [ ]:
# ============================================================
# STAGE 7 — CELL 2 (Option B): DPO without LoRA
# ============================================================
import torch
import time
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from trl import DPOConfig, DPOTrainer

# Free memory from previous stages
torch.cuda.empty_cache()

memory_before = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM before loading: {memory_before:.3f} GB")

# Load fresh SFT model as starting point for DPO
print("Loading SFT model for DPO...")
dpo_model = GPT2LMHeadModel.from_pretrained("./sft_model")
dpo_model = dpo_model.to("cuda")

# Count parameters
total = sum(p.numel() for p in dpo_model.parameters())
trainable = sum(p.numel() for p in dpo_model.parameters() if p.requires_grad)
print(f"Total parameters:     {total:,}")
print(f"Trainable parameters: {trainable:,}")
# All 124M are trainable — no LoRA
# DPOTrainer handles reference model internally

memory_after = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM after loading: {memory_after:.3f} GB")
print(f"VRAM free: {(14.56 - memory_after):.3f} GB")

# DPO Configuration
dpo_config = DPOConfig(
    output_dir="./dpo_model",

    learning_rate=1e-5,
    # Smaller than planned (was 1e-4 for LoRA)
    # Without LoRA all 124M parameters update
    # Need smaller steps to avoid destroying SFT structure

    per_device_train_batch_size=2,
    # Reduced from 4 to 2 — no LoRA means more memory per step
    # DPOTrainer needs policy + reference model simultaneously
    # batch size 2 keeps us safe on T4

    gradient_accumulation_steps=2,
    # Effective batch size = 2 × 2 = 4
    # Same math as SFT gradient accumulation trick
    # Process 2 examples, accumulate, process 2 more, update

    num_train_epochs=3,
    # 3 epochs — DPO is stable, same as SFT

    beta=0.1,
    # Beta controls how far policy moves from SFT reference
    # Same value as GRPO beta for fair comparison
    # Higher beta = stays closer to SFT
    # Lower beta = more aggressive preference learning

    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    seed=42,

    max_length=512,
    # Maximum combined length prompt + response

    max_prompt_length=256,
    # Prompt gets up to 256 tokens
    # Response gets remaining 256 tokens
)

print(f"\nDPO Configuration:")
print(f"  beta:                    {dpo_config.beta}")
print(f"  learning_rate:           {dpo_config.learning_rate}")
print(f"  batch_size:              {dpo_config.per_device_train_batch_size}")
print(f"  gradient_accumulation:   {dpo_config.gradient_accumulation_steps}")
print(f"  effective_batch_size:    {dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps}")
print(f"  epochs:                  {dpo_config.num_train_epochs}")
print(f"  max_length:              {dpo_config.max_length}")

# Create DPO Trainer
# DPOTrainer automatically creates a frozen reference copy
# of the model internally — we don't need to do this manually
dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_config,
    train_dataset=dpo_train,
    eval_dataset=dpo_val,
    processing_class=tokenizer_ppo,
)

print(f"\n✅ DPO Trainer created")
print(f"Train examples: {len(dpo_train)}")
print(f"Val examples:   {len(dpo_val)}")

memory_final = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM: {memory_final:.3f} GB used")
print(f"VRAM: {(14.56 - memory_final):.3f} GB free")

# ============================================================
# PSEUDOCODE SUMMARY:
# load fresh SFT model — all 124M parameters trainable
# no LoRA — full fine tuning
# DPOTrainer creates frozen reference copy internally
# configure DPO:
#   beta=0.1, lr=1e-5, batch_size=2, grad_accum=2
#   effective batch size = 4
#   3 epochs, max_length=512
# verify VRAM is sufficient before training
# ============================================================

VRAM before loading: 4.447 GB
Loading SFT model for DPO...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Total parameters:     124,439,808
Trainable parameters: 124,439,808
VRAM after loading: 4.911 GB
VRAM free: 9.649 GB


TypeError: DPOConfig.__init__() got an unexpected keyword argument 'max_prompt_length'

In [ ]:
# ============================================================
# DIAGNOSTIC: Check DPOConfig valid parameters
# ============================================================
from trl import DPOConfig
import inspect

sig = inspect.signature(DPOConfig.__init__)
params = list(sig.parameters.keys())

print("Valid DPOConfig parameters:")
for p in params:
    print(f"  {p}")

Valid DPOConfig parameters:
  self
  output_dir
  per_device_train_batch_size
  num_train_epochs
  max_steps
  learning_rate
  lr_scheduler_type
  lr_scheduler_kwargs
  warmup_steps
  optim
  optim_args
  weight_decay
  adam_beta1
  adam_beta2
  adam_epsilon
  optim_target_modules
  gradient_accumulation_steps
  average_tokens_across_devices
  max_grad_norm
  label_smoothing_factor
  bf16
  fp16
  bf16_full_eval
  fp16_full_eval
  tf32
  gradient_checkpointing
  gradient_checkpointing_kwargs
  torch_compile
  torch_compile_backend
  torch_compile_mode
  use_liger_kernel
  liger_kernel_config
  use_cache
  neftune_noise_alpha
  torch_empty_cache_steps
  auto_find_batch_size
  logging_strategy
  logging_steps
  logging_first_step
  log_on_each_node
  logging_nan_inf_filter
  include_num_input_tokens_seen
  log_level
  log_level_replica
  disable_tqdm
  report_to
  run_name
  project
  trackio_space_id
  trackio_bucket_id
  trackio_static_space_id
  eval_strategy
  eval_steps
  eval_delay

In [ ]:
# ============================================================
# STAGE 7 — CELL 2 FIXED: DPO Config with correct parameters
# ============================================================
from trl import DPOConfig, DPOTrainer

dpo_config = DPOConfig(
    output_dir="./dpo_model",
    learning_rate=1e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_train_epochs=3,
    beta=0.1,
    logging_steps=20,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none",
    seed=42,
    max_length=512,
    # max_prompt_length does not exist in TRL 1.6.0
    # max_length=512 handles total sequence length
    truncation_mode="keep_start",
    # keep_start = if sequence too long, keep beginning
    # alternative is keep_end
    # keep_start preserves the instruction context
)

print(f"DPO Configuration:")
print(f"  beta:                  {dpo_config.beta}")
print(f"  learning_rate:         {dpo_config.learning_rate}")
print(f"  batch_size:            {dpo_config.per_device_train_batch_size}")
print(f"  gradient_accumulation: {dpo_config.gradient_accumulation_steps}")
print(f"  effective_batch_size:  {dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps}")
print(f"  epochs:                {dpo_config.num_train_epochs}")
print(f"  max_length:            {dpo_config.max_length}")
print(f"  truncation_mode:       {dpo_config.truncation_mode}")

# Create DPO Trainer
dpo_trainer = DPOTrainer(
    model=dpo_model,
    args=dpo_config,
    train_dataset=dpo_train,
    eval_dataset=dpo_val,
    processing_class=tokenizer_ppo,
)

print(f"\n✅ DPO Trainer created")
print(f"Train examples: {len(dpo_train)}")
print(f"Val examples:   {len(dpo_val)}")

memory_final = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM: {memory_final:.3f} GB used")
print(f"VRAM: {(14.56 - memory_final):.3f} GB free")

# ============================================================
# PSEUDOCODE SUMMARY:
# fix: remove max_prompt_length (not valid in TRL 1.6.0)
# use max_length=512 and truncation_mode=keep_start instead
# create DPOTrainer with fixed config
# verify memory before training
# ============================================================

DPO Configuration:
  beta:                  0.1
  learning_rate:         1e-05
  batch_size:            2
  gradient_accumulation: 2
  effective_batch_size:  4
  epochs:                3
  max_length:            512
  truncation_mode:       keep_start


Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


✅ DPO Trainer created
Train examples: 900
Val examples:   100
VRAM: 5.376 GB used
VRAM: 9.184 GB free


In [ ]:
# ============================================================
# STAGE 7 — CELL 3: Run DPO training
# ============================================================
import time

print("Starting DPO training...")
print("=" * 55)
print("Watch every 20 steps:")
print("  loss          — DPO loss (should decrease)")
print("  rewards/chosen  — reward model score for chosen")
print("  rewards/rejected — reward model score for rejected")
print("  rewards/margins  — gap between chosen and rejected")
print("=" * 55)

start_time = time.time()

dpo_trainer.train()

end_time = time.time()
training_time = (end_time - start_time) / 60

print(f"\n✅ DPO Training Complete!")
print(f"Training time: {training_time:.1f} minutes")

# Save model
dpo_trainer.save_model("./dpo_model")
tokenizer_ppo.save_pretrained("./dpo_model")
print("Model saved to ./dpo_model")

memory_after = torch.cuda.memory_allocated(0) / (1024**3)
print(f"VRAM after training: {memory_after:.3f} GB")

# ============================================================
# PSEUDOCODE SUMMARY:
# start timer
# dpo_trainer.train() runs:
#     for each epoch (3 total):
#         for each batch of 2 pairs:
#             forward pass → get log probs for chosen and rejected
#             from both policy model and reference model
#             compute DPO loss:
#                 increase chosen probability relative to reference
#                 decrease rejected probability relative to reference
#                 beta controls how far from reference
#             backward pass → update weights
#         run validation → compute val loss
#         save checkpoint
# load best checkpoint
# save final model
# ============================================================

Starting DPO training...
Watch every 20 steps:
  loss          — DPO loss (should decrease)
  rewards/chosen  — reward model score for chosen
  rewards/rejected — reward model score for rejected
  rewards/margins  — gap between chosen and rejected


Epoch,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
1,0.610484,0.598922,2.047157,580380.000000,-94.264921,-97.669316,0.551635,-0.144330,-0.628163,0.605769,0.483833,-398.000357,-277.609167
2,0.262324,0.607836,1.886320,1160760.000000,-99.249131,-103.610125,0.548310,-0.841425,-1.529110,0.586538,0.687685,-404.971315,-286.618642


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
import torch
import shutil
from google.colab import drive

drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/deccan_ai_project', exist_ok=True)

try:
    shutil.copytree('./dpo_model',
        '/content/drive/MyDrive/deccan_ai_project/dpo_model',
        dirs_exist_ok=True)
    print("✅ dpo_model saved")
except Exception as e:
    print(f"⚠️ {e}")

try:
    shutil.copytree('./grpo_model',
        '/content/drive/MyDrive/deccan_ai_project/grpo_model',
        dirs_exist_ok=True)
    print("✅ grpo_model saved")
except Exception as e:
    print(f"⚠️ {e}")

try:
    torch.save(reward_model.state_dict(),
        '/content/drive/MyDrive/deccan_ai_project/reward_model.pt')
    print("✅ reward_model saved")
except Exception as e:
    print(f"⚠️ {e}")

try:
    shutil.copytree('./sft_model',
        '/content/drive/MyDrive/deccan_ai_project/sft_model',
        dirs_exist_ok=True)
    print("✅ sft_model saved")
except Exception as e:
    print(f"⚠️ {e}")

print("\n✅ Emergency save complete")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ dpo_model saved
✅ grpo_model saved
✅ reward_model saved
✅ sft_model saved

✅ Emergency save complete


## Stage 8 — Final Comparison Table

### All Real Numbers From This Project

| Metric                        | SFT        | GRPO       | DPO        |
|-------------------------------|------------|------------|------------|
| Training time                 | 46.5 min   | 11.5 min   | ~27 min    |
| Training examples             | 1800       | 300 prompts| 900 pairs  |
| Val loss / Val accuracy       | 2.304      | —          | 0.599      |
| Reward score (test prompts)   | -2.144     | -1.957     | —          |
| Rewards/margins               | —          | —          | 0.484-0.688|
| Rewards/accuracies            | —          | —          | 58.7-60.6% |
| Phrase bias "sure"            | 1/20       | 5/20       | —          |
| Reward hacking observed       | No         | Yes (mild) | No         |
| Needs reward model            | No         | Yes        | No         |
| Needs RL loop                 | No         | Yes        | No         |
| Memory requirement            | 1 model    | 2 models   | 2 models   |

### Key Findings

| Finding                       | Result                                    |
|-------------------------------|-------------------------------------------|
| Reward model accuracy         | 56.5% (58.7% val)                        |
| Main reward model bias        | Confident surface tone                    |
| GRPO vs SFT reward            | GRPO +0.19 mean (15/20 prompts)          |
| Reward hacking evidence       | "sure" 5x more frequent after GRPO       |
| KL removal effect             | Training unstable, loss spike 50.58      |
| KL finding                    | Stabilizes optimization, not just drift  |
| DPO rewards margin            | 0.484 (epoch 1) → 0.688 (epoch 2)       |
| DPO overfitting               | Mild — val loss up at epoch 2            |

I built a complete post-training pipeline on GPT-2 Small using UltraFeedback. Everything — SFT, reward model, GRPO, DPO — uses the same dataset throughout. No format mismatches.
I started by extracting 35,002 strong preference pairs from UltraFeedback's 63,967 examples, filtering to pairs with score gap greater than 2.0. I split cleanly into 2000 for SFT, 1000 for reward model and DPO, 200 held out for evaluation with zero overlap between splits.
SFT on 2000 chosen responses took 46.5 minutes. Validation loss reached 2.304 with 53.4% token accuracy. The model learned the instruction-following format — confirmed by comparing raw GPT-2 versus SFT GPT-2 on identical prompts. Output quality was limited by GPT-2 Small's 124M parameter capacity, which is expected.
I trained a reward model from scratch using DistilBERT with a linear reward head on 900 preference pairs. Bradley-Terry loss pushes chosen scores above rejected scores. Final validation accuracy was 58.7% — 8.7 points above random. Before using it in RL training, I ran deep evaluation on 200 held-out pairs. I found that accuracy dropped to 49% on medium-difficulty pairs — near random. Reading the failure cases revealed the real problem: the reward model learned confident surface tone over actual correctness.

 It preferred responses starting with 'Sure!' and penalized honest refusals. These became my predictions for what GRPO would exploit.

GRPO training on 300 prompts confirmed those predictions. On 20 test prompts, GRPO scored higher than SFT on 15 out of 20 — mean reward improvement of +0.19.

 But phrase bias analysis showed 'sure' appearing 5 times more frequently in GRPO outputs than SFT. Human-visible quality degraded.

 The policy successfully optimized the reward model's biases while producing worse outputs.

 This is reward hacking — mild but documented with real numbers.

For the KL experiment I ran GRPO with beta=0.0 versus beta=0.1.

Removing KL did not produce the textbook result of reward going up. Instead training destabilized — loss spike of 50.

58 at step 30 — and reward dropped further.

This revealed that KL serves two purposes: preventing policy drift AND stabilizing optimization against a noisy reward signal. Without KL, the policy chased noise instead of signal.


DPO on the same 1000 preference pairs trained without any reward model or RL loop.


 After 2 epochs, rewards margin reached 0.688 — chosen responses scoring 0.688 points higher than rejected on average. Rewards accuracy was 60.6% at epoch 1.

  DPO cannot be reward hacked the same way because there is no proxy reward to exploit — the preference signal is baked directly into the loss.

The most important insight from this project is about reward quality.

 Our reward model had systematic biases — not just random errors.
 GRPO amplified those biases. No amount of RL training fixes a broken reward signal.

